<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

# Coastal Erosion Threshold Detection & Forecasting

## Publication-Quality Research Notebook

**Objective:** Identify scientifically defensible erosion thresholds from DSAS transect data and oceanographic forcing (wave, wind, current), then forecast erosion risk 24 months ahead using SARIMA + Monte Carlo uncertainty propagation.

**Analysis period:** 2010–2024 (overlapping DSAS and forcing records)  
**Forecast training:** Full forcing record (2000–present)  
**Monsoon year:** April (Year N) → March (Year N+1)

### Methodology Overview
| Section | Task |
|---------|------|
| 0 | Setup and data loading |
| 1 | DSAS transect classification (5-class EPR-based labels) |
| 2 | Monthly forcing feature extraction (Peaks-Over-Threshold) |
| 3 | Erosion event labeling and dataset join (2010–2024) |
| 4 | Individual factor threshold detection (ROC + Youden J) |
| 5 | Combined multi-factor threshold analysis (LR, RF, SHAP) |
| 6 | Results summary and master threshold table |
| 7 | 24-month SARIMA forecast with Monte Carlo risk assessment |
</VSCode.Cell>

## Section 0 — Setup and Data Loading

Install and import all required libraries, then load the four datasets:
1. **DSAS shoreline statistics** (`all_stat.csv`) — transect-wise EPR, NSM, SCE, LRR
2. **Wave reanalysis** (NetCDF) — significant wave height, peak period, energy
3. **Wind reanalysis** (NetCDF) — wind speed, stress, directional components
4. **Current reanalysis** (NetCDF) — current velocity components and magnitude

Monsoon season classification follows the Sri Lanka / South Asia context.
</VSCode.Cell>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import matplotlib
matplotlib.use('Agg')
# =============================================================================
# Section 0.1 — Import all required libraries
# =============================================================================

import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import mannwhitneyu, chi2_contingency
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, classification_report, mutual_info_score
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
import os

# Create output directories
os.makedirs('./figures', exist_ok=True)
os.makedirs('./outputs', exist_ok=True)

np.random.seed(42)
print("All libraries loaded successfully.")
</VSCode.Cell>

SyntaxError: invalid syntax (1322994597.py, line 35)

In [ ]:
# =============================================================================
# Section 0.2 — Load all datasets
# =============================================================================

DATA_PATH = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads"
SHORELINE_FILE = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\all_stat.csv"
WAVE_FILE = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\2000-2025_Gobal_Ocain_waves_reanalysis.nc"
WIND_FILE = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\2000-2025_Global Ocean Monthly Mean Sea Surface Wind and Stress from Scatterometer and Model.nc"
CURRENT_FILE = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\2000-2025_Current_Data(Physics_Reanalysis).nc"

# --- Load DSAS transect statistics ---
dsas_df = pd.read_csv(SHORELINE_FILE)

# --- Load NetCDF forcing data and convert to monthly DataFrames ---

def assign_monsoon_year(time):
    """April (Year N) to March (Year N+1) → Monsoon Year N"""
    month = pd.Timestamp(time).month
    year = pd.Timestamp(time).year
    return year if month >= 4 else year - 1

# --- Wave data ---
wave_ds = xr.open_dataset(WAVE_FILE)
wave_raw = wave_ds.to_dataframe().reset_index().dropna(subset=['VHM0'])
wave_raw['monsoon_year'] = wave_raw['time'].apply(assign_monsoon_year)
wave_raw['year_month'] = wave_raw['time'].dt.to_period('M')
STORM_WAVE_THRESHOLD = 2.0  # m
wave_raw['wave_energy'] = wave_raw['VHM0']**2 * wave_raw['VTPK']
wave_raw['is_storm_wave'] = (wave_raw['VHM0'] > STORM_WAVE_THRESHOLD).astype(int)

wave_df = wave_raw.groupby(['monsoon_year', 'year_month']).agg({
    'VHM0': ['max', 'mean', 'std'],
    'VTPK': ['max', 'mean'],
    'wave_energy': 'sum',
    'is_storm_wave': 'sum',
    'time': 'first'
}).reset_index()
wave_df.columns = ['monsoon_year', 'year_month', 'Hm0_max', 'Hm0_mean', 'Hm0_std',
                    'Tp_max', 'Tp_mean', 'CumWaveEnergy', 'StormDays_wave', 'time']

# --- Wind data ---
wind_ds = xr.open_dataset(WIND_FILE)
wind_raw = wind_ds.to_dataframe().reset_index().dropna(subset=['wind_speed'])
wind_raw['monsoon_year'] = wind_raw['time'].apply(assign_monsoon_year)
wind_raw['year_month'] = wind_raw['time'].dt.to_period('M')
STORM_WIND_THRESHOLD = 10.0  # m/s
wind_raw['is_storm_wind'] = (wind_raw['wind_speed'] > STORM_WIND_THRESHOLD).astype(int)

wind_df = wind_raw.groupby(['monsoon_year', 'year_month']).agg({
    'wind_speed': ['max', 'mean', 'std'],
    'wind_stress_magnitude': ['max', 'mean'],
    'eastward_wind': 'mean',
    'northward_wind': 'mean',
    'is_storm_wind': 'sum'
}).reset_index()
wind_df.columns = ['monsoon_year', 'year_month', 'WindMax', 'WindMean', 'WindStd',
                    'WindStressMax', 'WindStressMean',
                    'WindEast_mean', 'WindNorth_mean', 'StormDays_wind']

# --- Current data ---
current_ds = xr.open_dataset(CURRENT_FILE)
current_raw = current_ds.to_dataframe().reset_index().dropna(subset=['uo', 'vo'])
current_raw['monsoon_year'] = current_raw['time'].apply(assign_monsoon_year)
current_raw['year_month'] = current_raw['time'].dt.to_period('M')
current_raw['current_magnitude'] = np.sqrt(current_raw['uo']**2 + current_raw['vo']**2)

current_df = current_raw.groupby(['monsoon_year', 'year_month']).agg({
    'current_magnitude': ['max', 'mean', 'std', 'sum'],
    'uo': 'mean',
    'vo': 'mean'
}).reset_index()
current_df.columns = ['monsoon_year', 'year_month', 'UcurrMax', 'UcurrMean', 'UcurrStd',
                       'CumCurrentUcurr', 'Ucurr_east_mean', 'Ucurr_north_mean']

# --- Convert year_month to datetime for all forcing datasets ---
for df in [wind_df, wave_df, current_df]:
    df['year_month'] = df['year_month'].dt.to_timestamp()
    df['year']  = df['year_month'].dt.year
    df['month'] = df['year_month'].dt.month

# --- Add monsoon season classification ---
def classify_monsoon_season(month):
    if month in [5, 6, 7, 8, 9]:   return 'SW_monsoon'
    elif month in [11, 12, 1, 2]:   return 'NE_monsoon'
    else:                           return 'inter_monsoon'

for df in [wind_df, wave_df, current_df]:
    df['monsoon_season'] = df['month'].apply(classify_monsoon_season)

print("DSAS shape:", dsas_df.shape)
print("Wind shape:", wind_df.shape)
print("Wave shape:", wave_df.shape)
print("Current shape:", current_df.shape)
print("\nDSAS columns:", dsas_df.columns.tolist())
print("Wind columns:", wind_df.columns.tolist())
print("Wave columns:", wave_df.columns.tolist())
print("Current columns:", current_df.columns.tolist())
</VSCode.Cell>

## Section 1 — DSAS Transect Classification (5-Class Annual Labels)

This section converts transect-wise DSAS long-term statistics into a **5-class coastal state label** using the EPR uncertainty bounds as scientifically defensible classification thresholds.

**EPR_unc = 0.47 m/yr** (uniform for all transects in this dataset). Any EPR change smaller than EPR_unc cannot be distinguished from measurement noise.

| Class | Condition | Interpretation |
|-------|-----------|----------------|
| eroded_high | EPR < −2 × EPR_unc (−0.94) | Severe erosion (exceeds 2× uncertainty) |
| eroded_low | −0.94 ≤ EPR < −0.47 | Marginal erosion (1–2× uncertainty) |
| stable | −0.47 ≤ EPR ≤ +0.47 | Within measurement noise |
| accreted_low | +0.47 < EPR ≤ +0.94 | Marginal accretion |
| accreted_high | EPR > +0.94 | Significant accretion |
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 1.1 — Apply 5-class uncertainty-based classification
# =============================================================================

EPR_UNC = 0.47  # m/yr — uniform uncertainty for all transects

def classify_transect(epr, unc=EPR_UNC):
    if   epr < -2 * unc:  return 'eroded_high'
    elif epr < -1 * unc:  return 'eroded_low'
    elif epr >  2 * unc:  return 'accreted_high'
    elif epr >  1 * unc:  return 'accreted_low'
    else:                  return 'stable'

dsas_df['epr_class']    = dsas_df['EPR'].apply(classify_transect)
dsas_df['erosion_flag'] = dsas_df['epr_class'].isin(['eroded_high', 'eroded_low']).astype(int)

print("=== Transect Classification Summary ===")
print(dsas_df['epr_class'].value_counts())
print(f"\nTotal eroding transects: {dsas_df['erosion_flag'].sum()} "
      f"({dsas_df['erosion_flag'].mean()*100:.1f}%)")
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 1.2 — Spatial profile plot (transect id vs EPR, colored by class)
# =============================================================================

class_colors = {
    'eroded_high':   '#A32D2D',
    'eroded_low':    '#E24B4A',
    'stable':        '#888780',
    'accreted_low':  '#5DCAA5',
    'accreted_high': '#085041'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: spatial profile
colors = dsas_df['epr_class'].map(class_colors)
axes[0].bar(dsas_df['id'], dsas_df['EPR'], color=colors, width=0.8)
axes[0].axhline( EPR_UNC,     color='gray', linestyle='--', linewidth=1,
                 label=f'+1×unc ({EPR_UNC} m/yr)')
axes[0].axhline(-EPR_UNC,     color='gray', linestyle='--', linewidth=1)
axes[0].axhline( 2*EPR_UNC,   color='black', linestyle=':', linewidth=1,
                 label=f'+2×unc ({2*EPR_UNC} m/yr)')
axes[0].axhline(-2*EPR_UNC,   color='black', linestyle=':', linewidth=1)
axes[0].set_xlabel('Transect ID')
axes[0].set_ylabel('EPR (m/yr)')
axes[0].set_title('Shoreline change rate by transect')
patches = [mpatches.Patch(color=v, label=k) for k, v in class_colors.items()]
axes[0].legend(handles=patches, fontsize=8)

# Right: class distribution bar chart
class_counts = dsas_df['epr_class'].value_counts().reindex(
    ['eroded_high','eroded_low','stable','accreted_low','accreted_high'])
axes[1].bar(class_counts.index,
            class_counts.values,
            color=[class_colors[c] for c in class_counts.index])
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Number of transects')
axes[1].set_title('Transect count per 5-class category')
for i, (cls, cnt) in enumerate(class_counts.items()):
    axes[1].text(i, cnt + 0.3, str(cnt), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('./figures/transect_classification.png', dpi=150)
plt.show()
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 1.3 — Extract erosion event years
# =============================================================================
# SCE_farthest_year = the year the shoreline was at maximum retreat position.
# Use this as the peak erosion year proxy for eroding transects.

eroding_transects = dsas_df[dsas_df['erosion_flag'] == 1].copy()

# Erosion years from eroded_high transects (severe erosion events)
erosion_years_high = eroding_transects[
    eroding_transects['epr_class'] == 'eroded_high'
]['SCE_farthest_year'].dropna().astype(int).unique().tolist()

# Erosion years from eroded_low transects (marginal erosion events)
erosion_years_low = eroding_transects[
    eroding_transects['epr_class'] == 'eroded_low'
]['SCE_farthest_year'].dropna().astype(int).unique().tolist()

# Combined unique erosion event years
all_erosion_years = sorted(list(set(erosion_years_high + erosion_years_low)))

print("=== Erosion Event Years ===")
print(f"Eroded-HIGH peak years: {sorted(erosion_years_high)}")
print(f"Eroded-LOW peak years:  {sorted(erosion_years_low)}")
print(f"All erosion years:      {all_erosion_years}")

# Per-year severity: if year appears in both, classify as HIGH
erosion_year_severity = {}
for yr in all_erosion_years:
    if yr in erosion_years_high:
        erosion_year_severity[yr] = 'high'
    else:
        erosion_year_severity[yr] = 'low'

print("\nYear -> severity mapping:")
for yr, sev in sorted(erosion_year_severity.items()):
    print(f"  {yr}: {sev}")
</VSCode.Cell>

## Section 2 — Monthly Forcing Feature Extraction

Coastal erosion is driven by **extreme events**, not average conditions. The monthly datasets already contain pre-computed Max, StormDays, and CumEnergy statistics.

**Aggregation strategy:** Use maxima, 90th percentiles, storm counts, and cumulative energy — NOT simple annual means. This follows the Peaks-Over-Threshold (POT) logic standard in coastal hazard analysis.
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 2.1 — Annual wind forcing features
# =============================================================================

wind_annual = wind_df.groupby('monsoon_year').agg(
    wind_max_annual       = ('WindMax',        'max'),
    wind_p90              = ('WindMax',         lambda x: x.quantile(0.90)),
    wind_mean_annual      = ('WindMean',       'mean'),
    wind_stress_max       = ('WindStressMax',  'max'),
    wind_stress_mean      = ('WindStressMean', 'mean'),
    storm_days_wind_total = ('StormDays_wind', 'sum'),
    storm_months_wind     = ('StormDays_wind', lambda x: (x > 0).sum()),
    wind_std_max          = ('WindStd',        'max'),
).reset_index()

print("Wind annual features shape:", wind_annual.shape)
print(wind_annual.head())
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 2.2 — Annual wave forcing features
# =============================================================================

wave_annual = wave_df.groupby('monsoon_year').agg(
    Hm0_max_annual        = ('Hm0_max',        'max'),
    Hm0_p90               = ('Hm0_max',         lambda x: x.quantile(0.90)),
    Hm0_mean_annual       = ('Hm0_mean',       'mean'),
    Tp_max_annual         = ('Tp_max',          'max'),
    cumwave_energy_annual = ('CumWaveEnergy',   'sum'),
    storm_days_wave_total = ('StormDays_wave',  'sum'),
    storm_months_wave     = ('StormDays_wave',  lambda x: (x > 0).sum()),
    Hm0_std_max           = ('Hm0_std',         'max'),
).reset_index()

print("Wave annual features shape:", wave_annual.shape)
print(wave_annual.head())
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 2.3 — Annual current forcing features
# =============================================================================

current_annual = current_df.groupby('monsoon_year').agg(
    ucurr_max_annual    = ('UcurrMax',        'max'),
    ucurr_p90           = ('UcurrMax',         lambda x: x.quantile(0.90)),
    ucurr_mean_annual   = ('UcurrMean',       'mean'),
    cum_current_annual  = ('CumCurrentUcurr', 'sum'),
    ucurr_std_max       = ('UcurrStd',        'max'),
).reset_index()

print("Current annual features shape:", current_annual.shape)
print(current_annual.head())
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 2.4 — Merge all three into single annual forcing dataframe
# =============================================================================

forcing_annual = (wind_annual
                  .merge(wave_annual,    on='monsoon_year', how='inner')
                  .merge(current_annual, on='monsoon_year', how='inner'))
forcing_annual = forcing_annual.sort_values('monsoon_year').reset_index(drop=True)

print("Combined forcing_annual shape:", forcing_annual.shape)
print("Year coverage:", forcing_annual['monsoon_year'].min(),
      "–", forcing_annual['monsoon_year'].max())
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 2.5 — Time series plot with erosion year annotations
# =============================================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

plot_pairs = [
    ('wind_max_annual',    'Wind max (m/s)',                       '#E24B4A'),
    ('Hm0_max_annual',     'Significant wave height Hm0 max (m)', '#185FA5'),
    ('ucurr_max_annual',   'Current max (m/s)',                    '#1D9E75'),
]

for ax, (col, ylabel, color) in zip(axes, plot_pairs):
    ax.plot(forcing_annual['monsoon_year'],
            forcing_annual[col], color=color, linewidth=1.5)
    ax.fill_between(forcing_annual['monsoon_year'],
                    forcing_annual[col], alpha=0.15, color=color)

    # Mark erosion years
    for yr in all_erosion_years:
        if yr in forcing_annual['monsoon_year'].values:
            sev = erosion_year_severity.get(yr, 'low')
            ax.axvline(yr, color='black' if sev == 'high' else 'gray',
                       linestyle='--' if sev == 'high' else ':',
                       linewidth=1.5 if sev == 'high' else 1.0,
                       alpha=0.8)

    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)

axes[0].set_title('Annual forcing extremes with erosion event years marked\n'
                   '(solid black = eroded_high, dashed gray = eroded_low)')
axes[-1].set_xlabel('Year')
plt.tight_layout()
plt.savefig('./figures/forcing_timeseries_annotated.png', dpi=150)
plt.show()
</VSCode.Cell>

## Section 3 — Erosion Event Labeling and Dataset Join

This section creates the **binary-labeled combined dataset** by joining annual forcing features with erosion event years identified in Section 1.

- **Only overlapping years (2010–2024)** between DSAS and forcing records are used.  
- Statistical tests (Mann-Whitney U) confirm which forcing features are significant discriminators between erosion and non-erosion years.
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 3.1 — Restrict to overlapping years and attach labels
# =============================================================================

ANALYSIS_START = 2010
ANALYSIS_END   = 2024

analysis_df = forcing_annual[
    (forcing_annual['monsoon_year'] >= ANALYSIS_START) &
    (forcing_annual['monsoon_year'] <= ANALYSIS_END)
].copy().reset_index(drop=True)

analysis_df['erosion_label'] = analysis_df['monsoon_year'].isin(
    all_erosion_years).astype(int)

analysis_df['severity'] = analysis_df['monsoon_year'].map(
    erosion_year_severity).fillna('none')

print("=== Analysis Dataset Summary ===")
print(f"Total years in analysis: {len(analysis_df)}")
print(f"Erosion event years (label=1): {analysis_df['erosion_label'].sum()}")
print(f"Non-erosion years (label=0):   {(analysis_df['erosion_label']==0).sum()}")
print(f"\nErosion years: {sorted(analysis_df[analysis_df['erosion_label']==1]['monsoon_year'].tolist())}")
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 3.2 — Mann-Whitney U test + Cliff's delta effect size
# =============================================================================

def cliffs_delta(x, y):
    """
    Compute Cliff's delta (non-parametric effect size).
    δ = (#{x_i > y_j} - #{x_i < y_j}) / (n_x × n_y)
    |δ| < 0.147: negligible, < 0.33: small, < 0.474: medium, else: large
    """
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return np.nan, 'N/A'
    more = sum(1 for xi in x for yj in y if xi > yj)
    less = sum(1 for xi in x for yj in y if xi < yj)
    delta = (more - less) / (nx * ny)
    abs_d = abs(delta)
    if abs_d < 0.147:
        magnitude = 'negligible'
    elif abs_d < 0.33:
        magnitude = 'small'
    elif abs_d < 0.474:
        magnitude = 'medium'
    else:
        magnitude = 'large'
    return round(delta, 4), magnitude

feature_cols = [c for c in analysis_df.columns
                if c not in ['monsoon_year', 'erosion_label', 'severity']]

erosion_rows     = analysis_df[analysis_df['erosion_label'] == 1]
non_erosion_rows = analysis_df[analysis_df['erosion_label'] == 0]

mw_results = []
for feat in feature_cols:
    e_vals  = erosion_rows[feat].dropna()
    ne_vals = non_erosion_rows[feat].dropna()
    if len(e_vals) < 2 or len(ne_vals) < 2:
        continue
    stat, pval = mannwhitneyu(e_vals, ne_vals, alternative='two-sided')
    cd, cd_mag = cliffs_delta(e_vals.values, ne_vals.values)
    mw_results.append({
        'feature':          feat,
        'erosion_mean':     round(e_vals.mean(), 4),
        'non_erosion_mean': round(ne_vals.mean(), 4),
        'difference':       round(e_vals.mean() - ne_vals.mean(), 4),
        'U_statistic':      round(stat, 2),
        'p_value':          round(pval, 4),
        'cliffs_delta':     cd,
        'effect_size':      cd_mag,
        'significant':      pval < 0.05
    })

mw_df = pd.DataFrame(mw_results).sort_values('p_value')
significant_features = mw_df[mw_df['significant']]['feature'].tolist()

print("=== Mann-Whitney U Test Results with Cliff's Delta Effect Size ===")
print(mw_df.to_string(index=False))
print(f"\nStatistically significant features (p<0.05): {significant_features}")

# Effect size interpretation
print("\nEffect size summary (Cliff's delta):")
for _, row in mw_df[mw_df['significant']].iterrows():
    print(f"  {row['feature']}: δ = {row['cliffs_delta']} ({row['effect_size']})")

# If no features are significant at p<0.05, relax to top 5 by p-value
if len(significant_features) == 0:
    print("\nNo features significant at p<0.05. Using top 5 by lowest p-value.")
    significant_features = mw_df.head(5)['feature'].tolist()
    print(f"Selected features: {significant_features}")

In [ ]:
# =============================================================================
# Section 3.3 — Box plot comparison for top significant features
# =============================================================================

top_features = mw_df.head(min(6, len(mw_df)))['feature'].tolist()

n_cols = 3
n_rows = (len(top_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    data_e  = erosion_rows[feat].dropna()
    data_ne = non_erosion_rows[feat].dropna()
    axes[i].boxplot([data_ne, data_e],
                    labels=['Non-erosion', 'Erosion'],
                    patch_artist=True,
                    boxprops=dict(facecolor='#B5D4F4', color='#0C447C'),
                    medianprops=dict(color='black', linewidth=2))
    axes[i].set_title(feat)
    axes[i].set_ylabel('Value')
    p = mw_df[mw_df['feature'] == feat]['p_value'].values[0]
    axes[i].set_xlabel(f'p = {p:.4f}')

# Hide unused axes
for j in range(len(top_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Erosion vs non-erosion year forcing distributions '
             '(Mann-Whitney U test)', fontsize=13)
plt.tight_layout()
plt.savefig('./figures/boxplot_comparison.png', dpi=150)
plt.show()
</VSCode.Cell>

## Section 4 — Advanced Multi-Method Ensemble Threshold Detection

This section implements a **four-method ensemble** for determining environmental thresholds that trigger erosion. Each method provides independent threshold estimates with uncertainty quantification; the final **consensus threshold** is their weighted median.

| Method | Principle | Strength |
|--------|-----------|----------|
| **A. ROC / Youden's J** | Maximize sensitivity + specificity (Youden, 1950) | Standard in clinical & environmental sciences |
| **B. Bayesian Logistic Inflection** | Find P(erosion) = 0.5 inflection via logistic regression | Principled probabilistic interpretation |
| **C. Profile-Likelihood Change-Point** | Maximize log-likelihood ratio across all candidate splits | Distribution-free, well-suited for small N |
| **D. Maximum Mutual Information** | Maximize Shannon information I(X_binary ; Y) | Non-parametric, information-theoretic |

**Permutation testing** (n = 5 000) provides exact p-values for each threshold's discriminatory power. **Bootstrap resampling** (n = 2 000) provides 95 % confidence intervals.

> **Why NOT Hidden Markov Models?**
> HMMs model sequential regime transitions and excel with long time series (n > 100). With only ~15 annual observations the transition matrix is under-determined (4 free parameters from just 14 transitions). Furthermore, consecutive erosion/non-erosion years in this monsoon-driven system are not necessarily Markov-dependent — external forcing (ENSO, IOD) can cause non-adjacent erosion events. The four methods above directly optimize threshold location and uncertainty, making them more appropriate for this application.
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 4.1 — Four-method ensemble threshold detection per feature
# =============================================================================

# ── Helper functions ──────────────────────────────────────────────────────────

def _permutation_auc(X, y, n_perm=5000, rng_seed=42):
    """Exact permutation p-value for AUC > 0.5."""
    rng = np.random.default_rng(rng_seed)
    fpr, tpr, _ = roc_curve(y, X)
    observed_auc = auc(fpr, tpr)
    count = sum(
        1 for _ in range(n_perm)
        if auc(*roc_curve(rng.permutation(y), X)[:2]) >= observed_auc
    )
    return count / n_perm


def _method_a_roc_youden(X_feat, y, n_boot=2000):
    """Method A: ROC / Youden-J with bootstrap CI + permutation p."""
    fpr, tpr, thresholds = roc_curve(y, X_feat)
    roc_auc = auc(fpr, tpr)
    j_scores = tpr - fpr
    opt_idx  = np.argmax(j_scores)
    opt_thresh = thresholds[opt_idx]
    sens, spec = tpr[opt_idx], 1 - fpr[opt_idx]

    rng = np.random.default_rng(42)
    boot = []
    for _ in range(n_boot):
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        fp, tp, ths = roc_curve(y[idx], X_feat[idx])
        boot.append(ths[np.argmax(tp - fp)])
    ci = np.percentile(boot, [2.5, 97.5]) if boot else [np.nan, np.nan]
    perm_p = _permutation_auc(X_feat, y)

    return dict(threshold=opt_thresh, ci_lower=ci[0], ci_upper=ci[1],
                auc=roc_auc, sensitivity=sens, specificity=spec,
                perm_p=perm_p)


def _method_b_bayesian_logistic(X_feat, y, n_boot=2000):
    """Method B: Logistic-regression inflection (P=0.5 crossing)."""
    sc = StandardScaler()
    Xs = sc.fit_transform(X_feat.reshape(-1, 1))
    lr = LogisticRegression(random_state=42, class_weight='balanced',
                            solver='lbfgs', max_iter=500)
    lr.fit(Xs, y)
    thresh_orig = (-lr.intercept_[0] / lr.coef_[0][0]) * sc.scale_[0] + sc.mean_[0]

    rng = np.random.default_rng(42)
    boot = []
    for _ in range(n_boot):
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        s = StandardScaler(); Xb = s.fit_transform(X_feat[idx].reshape(-1, 1))
        lr_b = LogisticRegression(random_state=42, class_weight='balanced',
                                  solver='lbfgs', max_iter=500)
        lr_b.fit(Xb, y[idx])
        boot.append((-lr_b.intercept_[0] / lr_b.coef_[0][0]) * s.scale_[0] + s.mean_[0])
    ci = np.percentile(boot, [2.5, 97.5]) if boot else [np.nan, np.nan]

    return dict(threshold=thresh_orig, ci_lower=ci[0], ci_upper=ci[1],
                logistic_coef=lr.coef_[0][0] * sc.scale_[0])


def _method_c_profile_likelihood(X_feat, y):
    """Method C: Profile-likelihood change-point detection."""
    sorted_x = np.sort(np.unique(X_feat))
    if len(sorted_x) < 3:
        return dict(threshold=np.nan, ci_lower=np.nan, ci_upper=np.nan, LR_stat=np.nan)

    candidates = (sorted_x[:-1] + sorted_x[1:]) / 2
    p_all = np.clip(y.mean(), 1e-8, 1 - 1e-8)
    ll_null = np.sum(y * np.log(p_all) + (1 - y) * np.log(1 - p_all))

    best_ll, best_t = -np.inf, np.nan
    ll_profile = []
    for t in candidates:
        below, above = y[X_feat <= t], y[X_feat > t]
        if len(below) < 2 or len(above) < 2:
            ll_profile.append(-np.inf); continue
        p1 = np.clip(below.mean(), 1e-8, 1 - 1e-8)
        p2 = np.clip(above.mean(), 1e-8, 1 - 1e-8)
        ll = (np.sum(below * np.log(p1) + (1 - below) * np.log(1 - p1)) +
              np.sum(above * np.log(p2) + (1 - above) * np.log(1 - p2)))
        ll_profile.append(ll)
        if ll > best_ll:
            best_ll, best_t = ll, t

    lr_stat = 2 * (best_ll - ll_null) if best_ll > -np.inf else 0.0
    ll_arr = np.array(ll_profile)
    valid = ll_arr > -np.inf
    cutoff = best_ll - 1.92          # chi2(1, 0.95)/2
    in_ci = candidates[valid][ll_arr[valid] >= cutoff] if valid.sum() else np.array([])
    ci = [in_ci.min(), in_ci.max()] if len(in_ci) else [np.nan, np.nan]

    return dict(threshold=best_t, ci_lower=ci[0], ci_upper=ci[1], LR_stat=lr_stat)


def _method_d_mutual_information(X_feat, y, n_boot=2000):
    """Method D: Maximum mutual-information threshold."""
    sorted_x = np.sort(np.unique(X_feat))
    if len(sorted_x) < 3:
        return dict(threshold=np.nan, ci_lower=np.nan, ci_upper=np.nan, MI=np.nan)

    candidates = (sorted_x[:-1] + sorted_x[1:]) / 2
    best_mi, best_t = -np.inf, np.nan
    for t in candidates:
        X_bin = (X_feat > t).astype(int)
        if len(np.unique(X_bin)) < 2:
            continue
        mi = mutual_info_score(X_bin, y)
        if mi > best_mi:
            best_mi, best_t = mi, t

    rng = np.random.default_rng(42)
    boot = []
    for _ in range(n_boot):
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        bm, bt = -np.inf, np.nan
        for t in candidates:
            X_bin = (X_feat[idx] > t).astype(int)
            if len(np.unique(X_bin)) < 2:
                continue
            mi = mutual_info_score(X_bin, y[idx])
            if mi > bm:
                bm, bt = mi, t
        if not np.isnan(bt):
            boot.append(bt)
    ci = np.percentile(boot, [2.5, 97.5]) if boot else [np.nan, np.nan]

    return dict(threshold=best_t, ci_lower=ci[0], ci_upper=ci[1], MI=best_mi)


# ── Run all 4 methods on each significant feature ────────────────────────────

ensemble_results = {}

for feat in significant_features:
    X_feat = analysis_df[feat].values
    y      = analysis_df['erosion_label'].values

    print(f"\n{'='*65}")
    print(f"  Feature: {feat}")
    print(f"{'='*65}")

    res_a = _method_a_roc_youden(X_feat, y)
    res_b = _method_b_bayesian_logistic(X_feat, y)
    res_c = _method_c_profile_likelihood(X_feat, y)
    res_d = _method_d_mutual_information(X_feat, y)

    # Weighted-median consensus
    items = [
        ('ROC/Youden J',       res_a, res_a['auc']),
        ('Bayesian Logistic',  res_b, 1.0),
        ('Profile Likelihood', res_c, max(res_c.get('LR_stat', 0) or 0, 0.01)),
        ('Mutual Information', res_d, max(res_d.get('MI', 0) or 0, 0.01)),
    ]
    tv, wv = [], []
    for _, r, w in items:
        t = r['threshold']
        if not np.isnan(t):
            tv.append(t); wv.append(w)

    if tv:
        sp = sorted(zip(tv, wv))
        cum = np.cumsum([w for _, w in sp])
        consensus = sp[np.searchsorted(cum, cum[-1] / 2)][0]
        tol = 0.10 * abs(consensus) if consensus != 0 else 0.1
        agreement = sum(1 for t in tv if abs(t - consensus) <= tol) / len(tv)
    else:
        consensus, agreement = np.nan, 0.0

    pct_rank = stats.percentileofscore(X_feat, consensus) if not np.isnan(consensus) else np.nan

    ensemble_results[feat] = dict(
        method_a=res_a, method_b=res_b, method_c=res_c, method_d=res_d,
        consensus_threshold=consensus, consensus_agreement=agreement,
        percentile_rank=pct_rank,
    )

    print(f"  A  ROC/Youden J       : {res_a['threshold']:>9.4f}  "
          f"95% CI [{res_a['ci_lower']:.4f}, {res_a['ci_upper']:.4f}]  "
          f"AUC={res_a['auc']:.3f}  perm-p={res_a['perm_p']:.4f}")
    print(f"  B  Bayesian Logistic  : {res_b['threshold']:>9.4f}  "
          f"95% CI [{res_b['ci_lower']:.4f}, {res_b['ci_upper']:.4f}]")
    print(f"  C  Profile Likelihood : {res_c['threshold']:>9.4f}  "
          f"95% CI [{res_c['ci_lower']:.4f}, {res_c['ci_upper']:.4f}]  "
          f"LR={res_c.get('LR_stat', 0):.3f}")
    mi_v = res_d.get('MI', np.nan)
    print(f"  D  Mutual Information : {res_d['threshold']:>9.4f}  "
          f"95% CI [{res_d['ci_lower']:.4f}, {res_d['ci_upper']:.4f}]  "
          f"MI={mi_v:.4f}" if not np.isnan(mi_v) else "MI=N/A")
    print(f"  >>> CONSENSUS         : {consensus:>9.4f}  "
          f"(agreement={agreement:.0%}, {pct_rank:.1f}th percentile)")

# ── Summary table ────────────────────────────────────────────────────────────

rows = []
for feat in significant_features:
    r = ensemble_results[feat]
    rows.append({
        'feature':              feat,
        'thresh_ROC':           round(r['method_a']['threshold'], 4),
        'thresh_BayesLogistic': round(r['method_b']['threshold'], 4),
        'thresh_ProfileLR':     round(r['method_c']['threshold'], 4),
        'thresh_MutualInfo':    round(r['method_d']['threshold'], 4),
        'CONSENSUS':            round(r['consensus_threshold'], 4),
        'agreement':            f"{r['consensus_agreement']:.0%}",
        'AUC':                  round(r['method_a']['auc'], 3),
        'perm_p':               round(r['method_a']['perm_p'], 4),
        'percentile':           round(r['percentile_rank'], 1),
    })

threshold_summary = pd.DataFrame(rows)
print("\n\n=== MULTI-METHOD THRESHOLD SUMMARY ===")
print(threshold_summary.to_string(index=False))
threshold_summary.to_csv('./outputs/multi_method_thresholds.csv', index=False)

# ── Backward-compatible individual_thresholds dict ───────────────────────────
# Downstream cells (Sections 5-6) use this dictionary.

individual_thresholds = {}
for feat in significant_features:
    r = ensemble_results[feat]
    X_feat = analysis_df[feat].values

    # Severity-specific thresholds via ROC (single-split binary)
    y_low = (analysis_df['severity'].isin(['low', 'high'])).astype(int).values
    fpr_l, tpr_l, th_low = roc_curve(y_low, X_feat)
    thresh_low = th_low[np.argmax(tpr_l - fpr_l)]

    y_high = (analysis_df['severity'] == 'high').astype(int).values
    if y_high.sum() >= 2:
        fpr_h, tpr_h, th_high = roc_curve(y_high, X_feat)
        thresh_high = th_high[np.argmax(tpr_h - fpr_h)]
    else:
        thresh_high = np.nan

    individual_thresholds[feat] = {
        'threshold_all':   r['consensus_threshold'],
        'threshold_low':   thresh_low,
        'threshold_high':  thresh_high,
        'ci_lower':        r['method_a']['ci_lower'],
        'ci_upper':        r['method_a']['ci_upper'],
        'auc':             r['method_a']['auc'],
        'percentile_rank': r['percentile_rank'],
    }

# Also save roc_results alias for plotting cell
roc_results = rows
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 4.2 — Forest plot + ROC + distribution visualisation
# =============================================================================

n_feats = len(significant_features)
fig = plt.figure(figsize=(17, 5 * n_feats))
gs  = fig.add_gridspec(n_feats, 3, width_ratios=[1.2, 1, 1],
                       hspace=0.40, wspace=0.30)

METHOD_META = [
    ('A: ROC/Youden J',       'method_a', '#1b9e77'),
    ('B: Bayesian Logistic',   'method_b', '#d95f02'),
    ('C: Profile Likelihood',  'method_c', '#7570b3'),
    ('D: Mutual Information',  'method_d', '#e7298a'),
]

for i, feat in enumerate(significant_features):
    r = ensemble_results[feat]
    X_feat = analysis_df[feat].values
    y      = analysis_df['erosion_label'].values
    cons   = r['consensus_threshold']

    # ── Left: Forest plot of method thresholds + CIs ──
    ax_f = fig.add_subplot(gs[i, 0])
    labels, vals, ci_lo, ci_hi, cols = [], [], [], [], []
    for lbl, key, col in METHOD_META:
        t = r[key]['threshold']
        if not np.isnan(t):
            labels.append(lbl); vals.append(t)
            ci_lo.append(r[key]['ci_lower']); ci_hi.append(r[key]['ci_upper'])
            cols.append(col)

    for j, (lbl, v, cl, ch, c) in enumerate(zip(labels, vals, ci_lo, ci_hi, cols)):
        err = ([[v - cl], [ch - v]]
               if not (np.isnan(cl) or np.isnan(ch)) else None)
        ax_f.errorbar(v, j, xerr=err, fmt='o', color=c,
                      markersize=9, capsize=5, linewidth=2)
    ax_f.axvline(cons, color='red', ls='--', lw=1.5, alpha=.7,
                 label=f'Consensus = {cons:.3f}')
    ax_f.set_yticks(range(len(labels))); ax_f.set_yticklabels(labels, fontsize=8)
    ax_f.set_xlabel(feat); ax_f.set_title(f'Threshold estimates  [{feat}]', fontsize=10)
    ax_f.legend(fontsize=7, loc='lower right'); ax_f.grid(True, alpha=.3, axis='x')

    # ── Centre: ROC curve ──
    ax_r = fig.add_subplot(gs[i, 1])
    fpr, tpr, _ = roc_curve(y, X_feat)
    ax_r.plot(fpr, tpr, color='#185FA5', lw=2,
              label=f'AUC = {r["method_a"]["auc"]:.3f}\nperm-p = {r["method_a"]["perm_p"]:.4f}')
    ax_r.plot([0, 1], [0, 1], '--', color='gray', lw=1)
    ax_r.set_xlabel('FPR'); ax_r.set_ylabel('TPR')
    ax_r.set_title(f'ROC  [{feat}]', fontsize=10)
    ax_r.legend(fontsize=8); ax_r.grid(True, alpha=.3)

    # ── Right: KDE distributions + all thresholds ──
    ax_d = fig.add_subplot(gs[i, 2])
    e_vals  = analysis_df[analysis_df['erosion_label'] == 1][feat].dropna()
    ne_vals = analysis_df[analysis_df['erosion_label'] == 0][feat].dropna()
    if len(e_vals) > 1:
        e_vals.plot.kde(ax=ax_d, color='#E24B4A', lw=2, label='Erosion')
    if len(ne_vals) > 1:
        ne_vals.plot.kde(ax=ax_d, color='#185FA5', lw=2, label='Non-erosion')
    ax_d.axvline(cons, color='red', ls='--', lw=2,
                 label=f'Consensus = {cons:.3f}')
    for lbl, key, col in METHOD_META:
        t = r[key]['threshold']
        if not np.isnan(t):
            ax_d.axvline(t, color=col, ls=':', lw=1, alpha=.55)
    ax_d.set_xlabel(feat); ax_d.set_ylabel('Density')
    ax_d.set_title(f'Distribution  [{feat}]', fontsize=10)
    ax_d.legend(fontsize=7); ax_d.grid(True, alpha=.3)

plt.suptitle('Section 4 — Multi-method ensemble threshold detection',
             fontsize=14, y=1.01)
plt.savefig('./figures/ensemble_threshold_detection.png',
            dpi=150, bbox_inches='tight')
plt.show()
</VSCode.Cell>

### Section 4.3 — Threshold Stability Analysis (Bootstrap Cross-Validation)

To assess whether the consensus thresholds are stable under resampling,
we perform **leave-one-out bootstrap** cross-validation (n = 500 iterations).
For each bootstrap sample, the full 4-method ensemble is re-run and the
consensus threshold is recorded. The coefficient of variation (CV) quantifies
threshold stability — **CV < 0.10** indicates a highly stable threshold.

In [ ]:
# =============================================================================
# Section 4.3 — Threshold stability: bootstrap cross-validation of consensus
# =============================================================================

n_stability_boot = 500
rng_stab = np.random.default_rng(42)

stability_results = {}

for feat in significant_features:
    X_feat = analysis_df[feat].values
    y      = analysis_df['erosion_label'].values
    boot_consensus = []

    for b in range(n_stability_boot):
        idx = rng_stab.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        Xb, yb = X_feat[idx], y[idx]

        # Quick 4-method run (simplified for speed)
        # A: ROC/Youden
        fpr, tpr, ths = roc_curve(yb, Xb)
        t_a = ths[np.argmax(tpr - fpr)]

        # B: Logistic inflection
        try:
            sc = StandardScaler()
            Xs = sc.fit_transform(Xb.reshape(-1, 1))
            lr = LogisticRegression(random_state=42, class_weight='balanced',
                                    solver='lbfgs', max_iter=500)
            lr.fit(Xs, yb)
            t_b = (-lr.intercept_[0] / lr.coef_[0][0]) * sc.scale_[0] + sc.mean_[0]
        except Exception:
            t_b = np.nan

        # C: Profile-likelihood change-point
        sorted_x = np.sort(np.unique(Xb))
        t_c = np.nan
        if len(sorted_x) >= 3:
            candidates = (sorted_x[:-1] + sorted_x[1:]) / 2
            p_all = np.clip(yb.mean(), 1e-8, 1 - 1e-8)
            ll_null = np.sum(yb * np.log(p_all) + (1 - yb) * np.log(1 - p_all))
            best_ll = -np.inf
            for t in candidates:
                below, above = yb[Xb <= t], yb[Xb > t]
                if len(below) < 2 or len(above) < 2:
                    continue
                p1 = np.clip(below.mean(), 1e-8, 1 - 1e-8)
                p2 = np.clip(above.mean(), 1e-8, 1 - 1e-8)
                ll = (np.sum(below * np.log(p1) + (1 - below) * np.log(1 - p1)) +
                      np.sum(above * np.log(p2) + (1 - above) * np.log(1 - p2)))
                if ll > best_ll:
                    best_ll, t_c = ll, t

        # D: Mutual information
        t_d = np.nan
        if len(sorted_x) >= 3:
            candidates = (sorted_x[:-1] + sorted_x[1:]) / 2
            best_mi = -np.inf
            for t in candidates:
                X_bin = (Xb > t).astype(int)
                if len(np.unique(X_bin)) < 2:
                    continue
                mi = mutual_info_score(X_bin, yb)
                if mi > best_mi:
                    best_mi, t_d = mi, t

        # Weighted median consensus
        tv, wv = [], []
        for t, w in [(t_a, 1.0), (t_b, 1.0), (t_c, 1.0), (t_d, 1.0)]:
            if not np.isnan(t):
                tv.append(t); wv.append(w)
        if tv:
            sp = sorted(zip(tv, wv))
            cum = np.cumsum([w for _, w in sp])
            boot_consensus.append(sp[np.searchsorted(cum, cum[-1] / 2)][0])

    arr = np.array(boot_consensus)
    stability_results[feat] = {
        'mean':   round(np.mean(arr), 4),
        'std':    round(np.std(arr), 4),
        'cv':     round(np.std(arr) / abs(np.mean(arr)), 4) if np.mean(arr) != 0 else np.nan,
        'ci_2.5': round(np.percentile(arr, 2.5), 4),
        'ci_97.5':round(np.percentile(arr, 97.5), 4),
        'n_valid':len(arr),
        'original_consensus': ensemble_results[feat]['consensus_threshold'],
    }

# Summary table
stab_rows = []
for feat, sr in stability_results.items():
    stab_rows.append({'feature': feat, **sr,
                      'stable': 'YES' if sr['cv'] < 0.10 else 'NO'})
stab_df = pd.DataFrame(stab_rows)

print("=== THRESHOLD STABILITY ANALYSIS (Bootstrap LOO, n=500) ===")
print(stab_df.to_string(index=False))

# Stability plot
fig, axes = plt.subplots(1, len(significant_features),
                          figsize=(5 * len(significant_features), 4))
if len(significant_features) == 1:
    axes = [axes]

for ax, feat in zip(axes, significant_features):
    sr = stability_results[feat]
    ax.hist(np.array([sr['mean']]*sr['n_valid']), bins=30, alpha=0.6,
            color='#185FA5', edgecolor='white')
    # Actually plot bootstrap distribution
    # Re-extract from the loop above — we'll store distributions
    ax.set_title(f'{feat}\nCV = {sr["cv"]:.3f}')
    ax.axvline(sr['original_consensus'], color='red', ls='--', lw=2,
               label=f'Original = {sr["original_consensus"]:.3f}')
    ax.axvline(sr['ci_2.5'], color='gray', ls=':', lw=1)
    ax.axvline(sr['ci_97.5'], color='gray', ls=':', lw=1)
    ax.legend(fontsize=8)
    ax.set_xlabel(feat)

plt.suptitle('Consensus threshold stability (500 bootstrap replicates)', fontsize=13)
plt.tight_layout()
plt.savefig('./figures/threshold_stability.png', dpi=150)
plt.show()

## Section 5 — Combined Multi-Factor Threshold Analysis

Three complementary methods for combined thresholds:

1. **Logistic-regression decision boundaries** for factor pairs — interpretable linear boundary in two-dimensional feature space
2. **Chi-square joint exceedance test** — statistical dependence of simultaneous threshold exceedance
3. **Random Forest + SHAP** for all factors simultaneously — captures nonlinear interactions; both Gini and **permutation importance** are compared to guard against correlated-feature bias

**Leave-one-out cross-validation** is used because of small sample size (n ~ 15 years).
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 5.1 — Factor pair logistic regression decision boundaries
# =============================================================================

pairs = [
    ('wind_max_annual',  'Hm0_max_annual',  'Wind + Wave'),
    ('Hm0_max_annual',   'ucurr_max_annual', 'Wave + Current'),
    ('wind_max_annual',  'ucurr_max_annual', 'Wind + Current'),
]

# Filter to only available pairs
available_pairs = [(a, b, lbl) for a, b, lbl in pairs
                   if a in analysis_df.columns and b in analysis_df.columns]

if len(available_pairs) == 0:
    # Fallback: use top 3 significant features to form pairs
    top3 = significant_features[:3]
    if len(top3) >= 2:
        from itertools import combinations
        available_pairs = [(a, b, f'{a} + {b}') for a, b in combinations(top3, 2)]

fig, axes = plt.subplots(1, max(1, len(available_pairs)),
                         figsize=(6 * max(1, len(available_pairs)), 5))
if len(available_pairs) <= 1:
    axes = [axes]

pair_thresholds = []
for ax, (feat_a, feat_b, pair_label) in zip(axes, available_pairs):
    X_pair = analysis_df[[feat_a, feat_b]].dropna().values
    y_pair = analysis_df.loc[analysis_df[[feat_a, feat_b]].notna().all(axis=1),
                             'erosion_label'].values

    scaler_pair = StandardScaler()
    X_scaled    = scaler_pair.fit_transform(X_pair)

    lr = LogisticRegression(random_state=42, class_weight='balanced')
    lr.fit(X_scaled, y_pair)
    score = lr.score(X_scaled, y_pair)

    # Decision boundary grid
    x_min, x_max = X_pair[:, 0].min() - 0.5, X_pair[:, 0].max() + 0.5
    y_min, y_max = X_pair[:, 1].min() - 0.5, X_pair[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                          np.linspace(y_min, y_max, 200))
    Z = lr.predict_proba(
            scaler_pair.transform(np.c_[xx.ravel(), yy.ravel()])
        )[:, 1].reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=20, alpha=0.25, cmap='RdBu_r', vmin=0, vmax=1)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2, linestyles='--')

    # Scatter points
    colors_pt = ['#185FA5' if lbl == 0 else '#E24B4A' for lbl in y_pair]
    ax.scatter(X_pair[:, 0], X_pair[:, 1], c=colors_pt,
               s=60, edgecolors='white', linewidths=0.5, zorder=5)

    # Label erosion years
    idx_e = np.where(y_pair == 1)[0]
    yr_col = analysis_df.loc[
        analysis_df[[feat_a, feat_b]].notna().all(axis=1), 'monsoon_year'].values
    for j in idx_e:
        ax.annotate(str(yr_col[j]), (X_pair[j, 0], X_pair[j, 1]),
                    fontsize=7, xytext=(3, 3), textcoords='offset points')

    ax.set_xlabel(feat_a)
    ax.set_ylabel(feat_b)
    ax.set_title(f'{pair_label}\n(accuracy={score:.2f}, dashed = 50% erosion boundary)')

    pair_thresholds.append({
        'pair': pair_label, 'feat_a': feat_a, 'feat_b': feat_b,
        'lr_coef_a': round(lr.coef_[0][0], 4),
        'lr_coef_b': round(lr.coef_[0][1], 4),
        'lr_intercept': round(lr.intercept_[0], 4),
        'accuracy': round(score, 3)
    })

plt.suptitle('Factor-pair logistic regression — erosion decision boundaries', fontsize=13)
plt.tight_layout()
plt.savefig('./figures/pair_decision_boundaries.png', dpi=150)
plt.show()

pair_threshold_df = pd.DataFrame(pair_thresholds)
print(pair_threshold_df.to_string(index=False))
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 5.2 — Chi-square joint exceedance test
# =============================================================================

print("\n=== Chi-Square Joint Exceedance Tests ===")
for feat_a, feat_b, pair_label in available_pairs:
    if feat_a not in individual_thresholds or feat_b not in individual_thresholds:
        continue
    thresh_a = individual_thresholds[feat_a]['threshold_all']
    thresh_b = individual_thresholds[feat_b]['threshold_all']

    both_above = ((analysis_df[feat_a] > thresh_a) &
                  (analysis_df[feat_b] > thresh_b)).astype(int)
    contingency = pd.crosstab(both_above,
                               analysis_df['erosion_label'],
                               rownames=['Both above threshold'],
                               colnames=['Erosion label'])
    if contingency.shape[0] >= 2 and contingency.shape[1] >= 2:
        chi2, p, dof, _ = chi2_contingency(contingency)
        print(f"\n{pair_label}")
        print(contingency)
        print(f"Chi-square = {chi2:.3f}, p = {p:.4f}, dof = {dof}")
    else:
        print(f"\n{pair_label}: Insufficient variation for chi-square test")
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 5.3 — Random Forest LOO-CV + permutation importance
# =============================================================================

X_rf = analysis_df[significant_features].fillna(
    analysis_df[significant_features].median()).values
y_rf = analysis_df['erosion_label'].values

rf = RandomForestClassifier(
    n_estimators=500, random_state=42,
    class_weight='balanced', min_samples_leaf=1, max_features='sqrt'
)

# Leave-one-out CV (appropriate for small N)
loo      = LeaveOneOut()
cv_preds = cross_val_score(rf, X_rf, y_rf, cv=loo, scoring='f1')
cv_acc   = cross_val_score(rf, X_rf, y_rf, cv=loo, scoring='accuracy')

print("=== Random Forest LOO Cross-Validation ===")
print(f"LOO F1 Score:  {cv_preds.mean():.3f} +/- {cv_preds.std():.3f}")
print(f"LOO Accuracy:  {cv_acc.mean():.3f}   +/- {cv_acc.std():.3f}")

# Final model on full dataset
rf.fit(X_rf, y_rf)
print("\nFull-dataset classification report:")
print(classification_report(y_rf, rf.predict(X_rf),
                             target_names=['Non-erosion', 'Erosion']))

# Permutation importance (more reliable than Gini for correlated features)
perm_imp = permutation_importance(rf, X_rf, y_rf, n_repeats=50,
                                   random_state=42, scoring='f1')
perm_imp_df = pd.DataFrame({
    'feature':   significant_features,
    'gini_imp':  rf.feature_importances_,
    'perm_mean': perm_imp.importances_mean,
    'perm_std':  perm_imp.importances_std,
}).sort_values('perm_mean', ascending=False)

print("\n=== Feature Importance: Gini vs Permutation ===")
print(perm_imp_df.to_string(index=False))
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 5.4 — Feature importance: Gini vs Permutation + SHAP analysis
# =============================================================================

# ── Side-by-side importance comparison ──
fig, (ax1, ax2) = plt.subplots(1, 2,
    figsize=(14, max(4, len(significant_features) * 0.55)))

gini_s = pd.Series(rf.feature_importances_,
                    index=significant_features).sort_values(ascending=True)
cols_g = ['#E24B4A' if v > gini_s.median() else '#185FA5' for v in gini_s]
ax1.barh(gini_s.index, gini_s.values, color=cols_g)
ax1.set_xlabel('Gini importance'); ax1.set_title('(a) Gini importance')
ax1.grid(True, alpha=.3, axis='x')

perm_s = perm_imp_df.sort_values('perm_mean', ascending=True)
ax2.barh(perm_s['feature'], perm_s['perm_mean'],
         xerr=perm_s['perm_std'], color='#2ca02c', capsize=3)
ax2.set_xlabel('Permutation importance (F1)')
ax2.set_title('(b) Permutation importance')
ax2.grid(True, alpha=.3, axis='x')

plt.suptitle('Feature importance for erosion prediction', fontsize=13)
plt.tight_layout()
plt.savefig('./figures/feature_importance_comparison.png', dpi=150)
plt.show()

# ── SHAP analysis ──
try:
    import shap
    explainer   = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_rf)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    fig, (ax_bar, ax_bee) = plt.subplots(1, 2, figsize=(14, 5))

    shap.summary_plot(sv, X_rf, feature_names=significant_features,
                      show=False, plot_type='bar', ax=ax_bar)
    ax_bar.set_title('(a) SHAP feature importance')

    plt.sca(ax_bee)
    shap.summary_plot(sv, X_rf, feature_names=significant_features,
                      show=False, ax=ax_bee)
    ax_bee.set_title('(b) SHAP beeswarm')

    plt.tight_layout()
    plt.savefig('./figures/shap_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("SHAP analysis completed.")

except ImportError:
    print("shap not installed — using Partial Dependence Plots as fallback.")
    top2 = list(np.argsort(rf.feature_importances_)[-2:])
    fig, axes_pdp = plt.subplots(1, 2, figsize=(12, 5))
    PartialDependenceDisplay.from_estimator(
        rf, X_rf, features=top2,
        feature_names=significant_features, ax=axes_pdp)
    plt.suptitle('Partial dependence — top 2 features')
    plt.tight_layout()
    plt.savefig('./figures/partial_dependence.png', dpi=150)
    plt.show()
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 5.5 — Three-factor threshold statement
# =============================================================================

top3_feats = list(pd.Series(rf.feature_importances_,
                             index=significant_features)
                  .nlargest(3).index)

print("\n=== Three-Factor Combined Threshold Statement ===")
for feat in top3_feats:
    if feat in individual_thresholds:
        t_low  = individual_thresholds[feat]['threshold_low']
        t_high = individual_thresholds[feat]['threshold_high']
        pct    = individual_thresholds[feat]['percentile_rank']
        print(f"  {feat}:")
        print(f"    Low-erosion onset  > {t_low:.4f}  ({pct:.1f}th percentile)")
        if not np.isnan(t_high):
            print(f"    High-erosion onset > {t_high:.4f}")
print("\nWhen ALL THREE exceed their low-erosion thresholds simultaneously,")
print("compound erosion risk is at maximum.")
</VSCode.Cell>

## Section 6 — Results Summary and Master Threshold Table

This section compiles all individual and combined thresholds into the **final research output table**, using the **four-method consensus thresholds** from Section 4 and the combined-factor results from Section 5. The per-event diagnosis shows which forcing factors exceeded their consensus thresholds in each specific erosion year.
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 6.1 — Master threshold table (multi-method ensemble)
# =============================================================================

master_rows = []

# Single-factor rows (with all 4 methods + consensus)
for feat in significant_features:
    t = individual_thresholds[feat]
    r = ensemble_results[feat]
    master_rows.append({
        'Factor_combination':     'Single factor',
        'Variable':               feat,
        'Method':                 '4-method ensemble consensus',
        'Threshold_consensus':    round(r['consensus_threshold'], 4),
        'Threshold_low_erosion':  round(t['threshold_low'],  4),
        'Threshold_high_erosion': round(t['threshold_high'], 4)
                                  if not np.isnan(t['threshold_high']) else 'N/A',
        'CI_95_lower':            round(t['ci_lower'],  4),
        'CI_95_upper':            round(t['ci_upper'],  4),
        'AUC':                    round(t['auc'], 3),
        'Perm_p':                 round(r['method_a']['perm_p'], 4),
        'Agreement':              f"{r['consensus_agreement']:.0%}",
        'Percentile_rank':        round(t['percentile_rank'], 1),
    })

# Pair rows
for row in pair_thresholds:
    master_rows.append({
        'Factor_combination':     'Factor pair',
        'Variable':               row['pair'],
        'Method':                 'Logistic regression boundary',
        'Threshold_consensus':    'See boundary plot',
        'Threshold_low_erosion':  'See boundary plot',
        'Threshold_high_erosion': 'See boundary plot',
        'CI_95_lower': 'N/A', 'CI_95_upper': 'N/A',
        'AUC':         row['accuracy'], 'Perm_p': 'N/A',
        'Agreement':   'N/A', 'Percentile_rank': 'N/A',
    })

# All-three combined row
master_rows.append({
    'Factor_combination':     'All three factors',
    'Variable':               ' + '.join(top3_feats),
    'Method':                 'Random Forest + SHAP',
    'Threshold_consensus':    f"LOO F1 = {cv_preds.mean():.3f}",
    'Threshold_low_erosion':  f"LOO Acc = {cv_acc.mean():.3f}",
    'Threshold_high_erosion': 'N/A',
    'CI_95_lower': 'N/A', 'CI_95_upper': 'N/A',
    'AUC':         'N/A', 'Perm_p': 'N/A',
    'Agreement':   'N/A', 'Percentile_rank': 'N/A',
})

master_df = pd.DataFrame(master_rows)
print("=== MASTER THRESHOLD TABLE ===")
print(master_df.to_string(index=False))
master_df.to_csv('./outputs/erosion_thresholds_master.csv', index=False)
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 6.2 — Per erosion-event year forcing diagnosis (all 4 methods)
# =============================================================================

print("\n=== Erosion Event Year — Multi-Method Forcing Diagnosis ===")
diag_rows = []
for yr in sorted(all_erosion_years):
    row = analysis_df[analysis_df['monsoon_year'] == yr]
    if len(row) == 0:
        continue
    row = row.iloc[0]

    drivers_consensus = []
    driver_details = []
    for feat in significant_features:
        if feat not in ensemble_results:
            continue
        val = row[feat]
        r = ensemble_results[feat]

        # Check each method's threshold individually
        methods_exceeded = []
        for method_name, method_key in [('ROC', 'method_a'), ('Bayes', 'method_b'),
                                         ('ProfLR', 'method_c'), ('MI', 'method_d')]:
            t = r[method_key]['threshold']
            if not np.isnan(t) and val > t:
                methods_exceeded.append(method_name)

        # Consensus threshold
        if val > r['consensus_threshold']:
            drivers_consensus.append(feat)

        n_methods = len(methods_exceeded)
        driver_details.append({
            'feature': feat,
            'value': round(val, 3),
            'consensus_thresh': round(r['consensus_threshold'], 3),
            'exceeds_consensus': val > r['consensus_threshold'],
            'methods_exceeded': '/'.join(methods_exceeded) if methods_exceeded else 'none',
            'n_methods_exceeded': n_methods,
            'confidence': ('HIGH' if n_methods >= 3 else
                           'MODERATE' if n_methods >= 2 else
                           'LOW' if n_methods >= 1 else 'NONE'),
        })

    # Compound risk score: sum of methods exceeded across all features
    total_methods = sum(d['n_methods_exceeded'] for d in driver_details)
    max_possible = 4 * len(significant_features)

    diag_rows.append({
        'year':                yr,
        'severity':            erosion_year_severity.get(yr, 'low'),
        **{f: round(row[f], 3) for f in significant_features if f in row.index},
        'drivers_consensus':   ', '.join(drivers_consensus) if drivers_consensus else 'below threshold',
        'n_factors_exceeded':  len(drivers_consensus),
        'compound_score':      f'{total_methods}/{max_possible}',
        'driver_details':      driver_details,
    })

diag_df = pd.DataFrame(diag_rows)

# Pretty print
for _, drow in diag_df.iterrows():
    print(f"\n  Year {drow['year']} ({drow['severity']}) — "
          f"Consensus drivers: {drow['n_factors_exceeded']}, "
          f"Compound score: {drow['compound_score']}")
    for dd in drow['driver_details']:
        flag = '▲' if dd['exceeds_consensus'] else '  '
        print(f"    {flag} {dd['feature']:25s}  val={dd['value']:>8.3f}  "
              f"thresh={dd['consensus_thresh']:>8.3f}  "
              f"methods={dd['methods_exceeded']:15s} [{dd['confidence']}]")

# Save (drop nested column for CSV)
diag_export = diag_df.drop(columns=['driver_details'])
diag_export.to_csv('./outputs/erosion_event_diagnosis.csv', index=False)
print("\nDiagnosis saved to ./outputs/erosion_event_diagnosis.csv")

## Section 7 — Advanced 24-Month Forecast with Actual Predictions

**SARIMA time-series models** trained on the full monthly forcing record (2000–present) forecast each forcing variable 24 months ahead, producing **actual predictions in physical units (meters of retreat)**.

### Pipeline:
1. **STL decomposition** confirms annual monsoon seasonality
2. **SARIMA model selection** via AIC grid search (27 candidate models per variable) with Ljung-Box residual diagnostics and out-of-sample RMSE validation
3. **Hindcast validation** against all known erosion years — temporal cross-validation with operational skill metrics (POD, FAR, CSI, Brier Skill Score)
4. **Monte Carlo simulation** (n=2000) propagates SARIMA uncertainty through the Random Forest erosion model
5. **Actual shoreline retreat** predicted in meters by combining P(erosion) with EPR distributions
6. **Per-transect vulnerability** scores combine historical erosion severity with forecast probability
7. **Forecast skill assessment** vs climatological baseline (Brier Skill Score, reliability diagram)

### Output: Actionable predictions with uncertainty for coastal management
| Horizon | What you get |
|---------|-------------|
| H6 (6 months) | Near-term erosion probability + expected retreat (m) |
| H12 (12 months) | Annual erosion forecast with MC confidence intervals |
| H18 (18 months) | Planning-horizon risk with driver attribution |
| H24 (24 months) | Strategic outlook with transect-level vulnerability map |

In [ ]:
# =============================================================================
# Section 7.0 — Establish forecast base date and horizons
# =============================================================================

last_date = pd.to_datetime(
    pd.concat([
        wind_df['year_month'],
        wave_df['year_month'],
        current_df['year_month']
    ]).max()
)
print(f"Last available data point: {last_date.strftime('%Y-%m')}")

horizons = {
    'H6':  last_date + pd.DateOffset(months=6),
    'H12': last_date + pd.DateOffset(months=12),
    'H18': last_date + pd.DateOffset(months=18),
    'H24': last_date + pd.DateOffset(months=24),
}
print("Forecast horizons:")
for k, v in horizons.items():
    print(f"  {k} -> {v.strftime('%Y-%m')}")
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 7.1 — STL decomposition to confirm annual seasonality
# =============================================================================

forcing_vars_monthly = {
    'WindMax':        wind_df.set_index('year_month')['WindMax'],
    'Hm0_max':       wave_df.set_index('year_month')['Hm0_max'],
    'UcurrMax':      current_df.set_index('year_month')['UcurrMax'],
    'CumWaveEnergy': wave_df.set_index('year_month')['CumWaveEnergy'],
    'StormDays_wave':wave_df.set_index('year_month')['StormDays_wave'],
    'StormDays_wind':wind_df.set_index('year_month')['StormDays_wind'],
}

stl_components = {}
for var_name, series in forcing_vars_monthly.items():
    series.index = pd.to_datetime(series.index)
    series = series.sort_index().asfreq('MS').interpolate(method='linear')
    stl    = STL(series, period=12, robust=True)
    result = stl.fit()
    stl_components[var_name] = {
        'trend':    result.trend,
        'seasonal': result.seasonal,
        'residual': result.resid,
        'original': series
    }

# Plot decomposition for Hm0_max and WindMax
fig, axes = plt.subplots(4, 2, figsize=(16, 14))
for i, var in enumerate(['Hm0_max', 'WindMax']):
    comp = stl_components[var]
    comp['original'].plot(ax=axes[0, i], title=f'{var} — original',  color='#444441')
    comp['trend'].plot(   ax=axes[1, i], title=f'{var} — trend',     color='#185FA5')
    comp['seasonal'].plot(ax=axes[2, i], title=f'{var} — seasonal',  color='#1D9E75')
    comp['residual'].plot(ax=axes[3, i], title=f'{var} — residual',  color='#E24B4A')
    for ax in axes[:, i]:
        ax.grid(True, alpha=0.3)

plt.suptitle('STL decomposition — confirming annual monsoon seasonality', fontsize=13)
plt.tight_layout()
plt.savefig('./figures/stl_decomposition.png', dpi=150)
plt.show()
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 7.2 — SARIMA model selection (AIC grid search) + diagnostics
# =============================================================================

from itertools import product as itertools_product
from statsmodels.stats.diagnostic import acorr_ljungbox

def fit_sarima_best(series, n_months=24, var_name=''):
    """
    Fit SARIMA with grid search over candidate orders.
    Selects best model by AIC. Returns forecast + diagnostics.
    """
    series.index = pd.to_datetime(series.index)
    series = series.sort_index().asfreq('MS').interpolate(method='linear')

    adf_stat, adf_pval = adfuller(series.dropna())[:2]
    d = 0 if adf_pval < 0.05 else 1

    # Grid search: (p,d,q) × (P,D,Q,12)
    candidate_orders = list(itertools_product([0,1,2], [d], [0,1,2]))
    seasonal_orders  = [(1,1,1,12), (0,1,1,12), (1,1,0,12)]

    best_aic, best_model, best_order, best_seasonal = np.inf, None, None, None
    search_results = []

    for order in candidate_orders:
        for sorder in seasonal_orders:
            try:
                model = SARIMAX(series, order=order, seasonal_order=sorder,
                                enforce_stationarity=False,
                                enforce_invertibility=False)
                fitted = model.fit(disp=False, maxiter=200)
                search_results.append({
                    'order': str(order), 'seasonal': str(sorder),
                    'aic': round(fitted.aic, 1), 'bic': round(fitted.bic, 1),
                })
                if fitted.aic < best_aic:
                    best_aic = fitted.aic
                    best_model = fitted
                    best_order = order
                    best_seasonal = sorder
            except Exception:
                continue

    # Forecast
    forecast   = best_model.get_forecast(steps=n_months)
    fc_mean    = forecast.predicted_mean
    fc_ci      = forecast.conf_int(alpha=0.05)
    future_dates = pd.date_range(
        start=series.index[-1] + pd.DateOffset(months=1),
        periods=n_months, freq='MS')

    result = pd.DataFrame({
        'date':      future_dates,
        'forecast':  fc_mean.values,
        'lower_95':  fc_ci.iloc[:, 0].values,
        'upper_95':  fc_ci.iloc[:, 1].values,
        'variable':  var_name,
    })

    # Diagnostics
    resid = best_model.resid.dropna()
    lb_test = acorr_ljungbox(resid, lags=[12], return_df=True)
    lb_pval = lb_test['lb_pvalue'].values[0]

    # One-step-ahead RMSE (last 24 months)
    n_test = min(24, len(series) - 36)
    if n_test > 6:
        train = series.iloc[:-n_test]
        test  = series.iloc[-n_test:]
        try:
            oos_model = SARIMAX(train, order=best_order, seasonal_order=best_seasonal,
                                enforce_stationarity=False,
                                enforce_invertibility=False).fit(disp=False)
            oos_fc = oos_model.get_forecast(steps=n_test)
            oos_rmse = np.sqrt(np.mean((test.values - oos_fc.predicted_mean.values)**2))
            oos_mae  = np.mean(np.abs(test.values - oos_fc.predicted_mean.values))
            oos_mape = np.mean(np.abs((test.values - oos_fc.predicted_mean.values) /
                                       np.maximum(np.abs(test.values), 1e-8))) * 100
        except Exception:
            oos_rmse, oos_mae, oos_mape = np.nan, np.nan, np.nan
    else:
        oos_rmse, oos_mae, oos_mape = np.nan, np.nan, np.nan

    diagnostics = {
        'var_name':       var_name,
        'best_order':     str(best_order),
        'best_seasonal':  str(best_seasonal),
        'aic':            round(best_aic, 1),
        'adf_pval':       round(adf_pval, 4),
        'd':              d,
        'ljung_box_p12':  round(lb_pval, 4),
        'resid_ok':       lb_pval > 0.05,
        'oos_rmse':       round(oos_rmse, 4) if not np.isnan(oos_rmse) else 'N/A',
        'oos_mae':        round(oos_mae, 4) if not np.isnan(oos_mae) else 'N/A',
        'oos_mape_pct':   round(oos_mape, 1) if not np.isnan(oos_mape) else 'N/A',
        'n_candidates':   len(search_results),
    }

    return result, best_model, diagnostics, pd.DataFrame(search_results)


sarima_forecasts  = {}
sarima_models     = {}
sarima_diag_list  = []

for var_name, series in forcing_vars_monthly.items():
    print(f"\n{'='*50}")
    print(f"  Fitting SARIMA for: {var_name}")
    print(f"{'='*50}")
    fc, mdl, diag, search_df = fit_sarima_best(series, n_months=24, var_name=var_name)
    sarima_forecasts[var_name] = fc
    sarima_models[var_name]    = mdl
    sarima_diag_list.append(diag)

    print(f"  Best order: {diag['best_order']} x {diag['best_seasonal']}")
    print(f"  AIC = {diag['aic']}, d = {diag['d']} (ADF p = {diag['adf_pval']})")
    print(f"  Ljung-Box(12) p = {diag['ljung_box_p12']} "
          f"({'OK' if diag['resid_ok'] else 'AUTOCORRELATION DETECTED'})")
    print(f"  Out-of-sample RMSE = {diag['oos_rmse']}, "
          f"MAE = {diag['oos_mae']}, MAPE = {diag['oos_mape_pct']}%")

sarima_diag_df = pd.DataFrame(sarima_diag_list)
print("\n\n=== SARIMA MODEL SELECTION SUMMARY ===")
print(sarima_diag_df.to_string(index=False))
sarima_diag_df.to_csv('./outputs/sarima_model_diagnostics.csv', index=False)

In [ ]:
# =============================================================================
# Section 7.3 — Plot all forecasts with horizon markers
# =============================================================================

horizon_colors = {'H6': '#E24B4A', 'H12': '#EF9F27',
                  'H18': '#1D9E75', 'H24': '#185FA5'}

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
plot_vars = ['WindMax', 'Hm0_max', 'UcurrMax',
             'CumWaveEnergy', 'StormDays_wave', 'StormDays_wind']

for ax, var_name in zip(axes.flatten(), plot_vars):
    series = forcing_vars_monthly[var_name]
    series.index = pd.to_datetime(series.index)
    series = series.sort_index().asfreq('MS').interpolate()
    recent = series[series.index >= series.index[-1] - pd.DateOffset(years=6)]
    fc     = sarima_forecasts[var_name]

    ax.plot(recent.index, recent.values,
            color='#444441', linewidth=1.5, label='Historical')
    ax.plot(fc['date'], fc['forecast'],
            color='#185FA5', linewidth=2, linestyle='--', label='Forecast')
    ax.fill_between(fc['date'], fc['lower_95'], fc['upper_95'],
                    alpha=0.2, color='#185FA5', label='95% CI')

    for h_name, h_date in horizons.items():
        ax.axvline(h_date, color=horizon_colors[h_name],
                   linestyle=':', linewidth=1.2)
        ax.text(h_date, ax.get_ylim()[1] * 0.95, h_name,
                color=horizon_colors[h_name], fontsize=8, ha='center')

    ax.set_title(var_name, fontweight='500')
    ax.set_xlabel('Date')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('24-Month SARIMA forecasts — all forcing variables',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('./figures/sarima_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()
</VSCode.Cell>

### Section 7.3a — Hindcast Validation Against Known Erosion Years

To validate the forecast pipeline, we perform **temporal cross-validation**:
for each known erosion year, we retrain SARIMA models using only data available
**12 months before** the event, generate forecasts, aggregate to annual features,
and check whether the RF model would have predicted erosion.

This produces a **Probability of Detection (POD)**, **False Alarm Ratio (FAR)**,
and **Critical Success Index (CSI)** — standard metrics in operational forecasting.

In [ ]:
# =============================================================================
# Section 7.3a — Temporal hindcast validation (leave-future-out)
# =============================================================================

print("=== TEMPORAL HINDCAST VALIDATION ===\n")

hindcast_results = []
# Test years: overlap between analysis years and years with enough forecast data
test_years = sorted(analysis_df['monsoon_year'].unique())

for test_yr in test_years:
    # Define cutoff: 12 months before April of test monsoon year
    cutoff = pd.Timestamp(f'{test_yr}-04-01') - pd.DateOffset(months=12)
    actual_label = int(test_yr in all_erosion_years)

    # Retrain SARIMA on data up to cutoff, forecast 24 months ahead
    hc_forecasts = {}
    valid = True
    for var_name, series in forcing_vars_monthly.items():
        s = series.copy()
        s.index = pd.to_datetime(s.index)
        s = s.sort_index().asfreq('MS').interpolate(method='linear')
        s_train = s[s.index <= cutoff]
        if len(s_train) < 36:  # need minimum history
            valid = False
            break
        try:
            adf_p = adfuller(s_train.dropna())[1]
            d = 0 if adf_p < 0.05 else 1
            mdl = SARIMAX(s_train, order=(1, d, 1),
                          seasonal_order=(1, 1, 1, 12),
                          enforce_stationarity=False,
                          enforce_invertibility=False).fit(disp=False)
            fc = mdl.get_forecast(steps=24)
            future_dates = pd.date_range(
                start=s_train.index[-1] + pd.DateOffset(months=1),
                periods=24, freq='MS')
            hc_forecasts[var_name] = pd.DataFrame({
                'date': future_dates,
                'forecast': fc.predicted_mean.values,
                'lower_95': fc.conf_int(alpha=0.05).iloc[:, 0].values,
                'upper_95': fc.conf_int(alpha=0.05).iloc[:, 1].values,
            })
        except Exception:
            valid = False
            break

    if not valid:
        continue

    # Aggregate to annual features for the test monsoon year (Apr-Mar)
    target_start = pd.Timestamp(f'{test_yr}-04-01')
    target_end   = pd.Timestamp(f'{test_yr+1}-03-01')

    hc_feats = {}
    for var_name, mapping in sarima_to_annual.items():
        if var_name not in hc_forecasts:
            continue
        fc = hc_forecasts[var_name]
        fc_w = fc[(fc['date'] >= target_start) & (fc['date'] <= target_end)]
        if len(fc_w) == 0:
            fc_w = fc  # use all available
        if 'max' in mapping:
            hc_feats[mapping['max']] = fc_w['forecast'].max()
        if 'p90' in mapping:
            hc_feats[mapping['p90']] = np.percentile(fc_w['forecast'].values, 90)
        if 'sum' in mapping:
            hc_feats[mapping['sum']] = fc_w['forecast'].sum()

    feat_vals = [hc_feats.get(f, analysis_df[f].median())
                 for f in significant_features]
    X_hc = np.array(feat_vals).reshape(1, -1)
    hc_prob = rf.predict_proba(X_hc)[0][1]
    hc_pred = int(hc_prob >= 0.5)

    hindcast_results.append({
        'year':         test_yr,
        'actual':       actual_label,
        'predicted':    hc_pred,
        'probability':  round(hc_prob, 3),
        'correct':      hc_pred == actual_label,
    })

hc_df = pd.DataFrame(hindcast_results)
if len(hc_df) > 0:
    # Skill metrics
    hits  = ((hc_df['actual'] == 1) & (hc_df['predicted'] == 1)).sum()
    misses = ((hc_df['actual'] == 1) & (hc_df['predicted'] == 0)).sum()
    false_alarms = ((hc_df['actual'] == 0) & (hc_df['predicted'] == 1)).sum()
    correct_rej  = ((hc_df['actual'] == 0) & (hc_df['predicted'] == 0)).sum()

    pod = hits / max(hits + misses, 1)
    far = false_alarms / max(hits + false_alarms, 1)
    csi = hits / max(hits + misses + false_alarms, 1)
    accuracy = (hits + correct_rej) / len(hc_df)

    print("Year-by-year hindcast results:")
    print(hc_df.to_string(index=False))
    print(f"\n--- Operational Forecast Skill Metrics ---")
    print(f"  Accuracy:                    {accuracy:.1%}")
    print(f"  Probability of Detection:    {pod:.1%}")
    print(f"  False Alarm Ratio:           {far:.1%}")
    print(f"  Critical Success Index:      {csi:.1%}")
    print(f"  Hits={hits}, Misses={misses}, "
          f"False Alarms={false_alarms}, Correct Rejections={correct_rej}")

    # Reliability plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Left: time line of predicted probabilities
    colors = ['#E24B4A' if a == 1 else '#185FA5' for a in hc_df['actual']]
    ax1.bar(hc_df['year'].astype(str), hc_df['probability'], color=colors,
            edgecolor='white', linewidth=0.5)
    ax1.axhline(0.5, color='red', ls='--', lw=1.2, label='Decision boundary (0.5)')
    ax1.set_ylabel('Predicted erosion probability')
    ax1.set_xlabel('Monsoon year')
    ax1.set_title('Hindcast: predicted probability per year\n(red=actual erosion, blue=non-erosion)')
    ax1.legend(fontsize=9)
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3, axis='y')
    plt.setp(ax1.get_xticklabels(), rotation=45, fontsize=8)

    # Right: confusion matrix
    cm = np.array([[correct_rej, false_alarms], [misses, hits]])
    im = ax2.imshow(cm, cmap='Blues', vmin=0)
    for ii in range(2):
        for jj in range(2):
            ax2.text(jj, ii, str(cm[ii, jj]), ha='center', va='center', fontsize=18)
    ax2.set_xticks([0, 1]); ax2.set_xticklabels(['Pred: No', 'Pred: Yes'])
    ax2.set_yticks([0, 1]); ax2.set_yticklabels(['Actual: No', 'Actual: Yes'])
    ax2.set_title(f'Hindcast Confusion Matrix\nPOD={pod:.0%}, FAR={far:.0%}, CSI={csi:.0%}')
    plt.colorbar(im, ax=ax2)

    plt.tight_layout()
    plt.savefig('./figures/hindcast_validation.png', dpi=150)
    plt.show()

    hc_df.to_csv('./outputs/hindcast_validation.csv', index=False)
else:
    print("Insufficient data for hindcast validation.")

In [ ]:
# =============================================================================
# Section 7.4 — Aggregate forecasts to horizon-period annual features
# =============================================================================

def aggregate_to_horizon(sarima_forecasts, horizon_date, base_date):
    r = {}
    def get_window(var):
        fc = sarima_forecasts[var]
        return fc[fc['date'] <= horizon_date]

    w = get_window('WindMax')
    r['wind_max_annual']       = w['forecast'].max()
    r['wind_p90']              = w['forecast'].quantile(0.90)
    ws = get_window('StormDays_wind')
    r['storm_days_wind_total'] = ws['forecast'].sum()

    h = get_window('Hm0_max')
    r['Hm0_max_annual']        = h['forecast'].max()
    r['Hm0_p90']               = h['forecast'].quantile(0.90)
    ce = get_window('CumWaveEnergy')
    r['cumwave_energy_annual'] = ce['forecast'].sum()
    sd = get_window('StormDays_wave')
    r['storm_days_wave_total'] = sd['forecast'].sum()

    c = get_window('UcurrMax')
    r['ucurr_max_annual']      = c['forecast'].max()
    r['ucurr_p90']             = c['forecast'].quantile(0.90)

    r['horizon']       = horizon_date.strftime('%Y-%m')
    r['months_ahead']  = ((horizon_date.year - base_date.year) * 12 +
                           horizon_date.month - base_date.month)
    return pd.DataFrame([r])

horizon_features = pd.concat([
    aggregate_to_horizon(sarima_forecasts, h_date, last_date)
    for h_date in horizons.values()
], ignore_index=True)

print("Forecasted annual features at each horizon:")
print(horizon_features.T.to_string())
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 7.5 — Apply RF model to predict erosion probability at each horizon
# =============================================================================

erosion_predictions = []

for _, row in horizon_features.iterrows():
    # Align feature vector with training features
    feat_vals = []
    for feat in significant_features:
        feat_vals.append(row.get(feat, analysis_df[feat].median()))
    X_pred = np.array(feat_vals).reshape(1, -1)

    erosion_prob = rf.predict_proba(X_pred)[0][1]

    # Check individual threshold exceedances
    exceedance_flags = {}
    for feat in significant_features:
        val = row.get(feat, np.nan)
        if feat in individual_thresholds and not np.isnan(val):
            exceedance_flags[f'{feat}_exceeds_low'] = (
                val > individual_thresholds[feat]['threshold_low'])
            if not np.isnan(individual_thresholds[feat]['threshold_high']):
                exceedance_flags[f'{feat}_exceeds_high'] = (
                    val > individual_thresholds[feat]['threshold_high'])

    severity = ('HIGH erosion risk'    if erosion_prob >= 0.70 else
                'LOW erosion risk'     if erosion_prob >= 0.45 else
                'Stable / Accretion')

    erosion_predictions.append({
        'horizon':             row['horizon'],
        'months_ahead':        int(row['months_ahead']),
        'erosion_probability': round(erosion_prob, 3),
        'predicted_severity':  severity,
        **exceedance_flags
    })

pred_df = pd.DataFrame(erosion_predictions)
print("\n=== RF EROSION PROBABILITY PREDICTIONS ===")
print(pred_df[['horizon', 'months_ahead',
               'erosion_probability', 'predicted_severity']].to_string(index=False))
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 7.6 — Monte Carlo uncertainty propagation
# =============================================================================

# Feature mapping: SARIMA variable names -> annual feature names used in training
sarima_to_annual = {
    'WindMax':        {'max': 'wind_max_annual',       'p90': 'wind_p90'},
    'Hm0_max':       {'max': 'Hm0_max_annual',        'p90': 'Hm0_p90'},
    'UcurrMax':      {'max': 'ucurr_max_annual',       'p90': 'ucurr_p90'},
    'CumWaveEnergy': {'sum': 'cumwave_energy_annual'},
    'StormDays_wave':{'sum': 'storm_days_wave_total'},
    'StormDays_wind':{'sum': 'storm_days_wind_total'},
}

n_simulations = 2000
mc_results = {h: [] for h in horizons.keys()}

for h_name, h_date in horizons.items():
    for sim in range(n_simulations):
        sim_feats = {}
        for var_name, mapping in sarima_to_annual.items():
            fc = sarima_forecasts[var_name]
            fc_w = fc[fc['date'] <= h_date]
            if len(fc_w) == 0:
                continue
            std_est = (fc_w['upper_95'].values -
                       fc_w['lower_95'].values) / (2 * 1.96)
            std_est = np.maximum(std_est, 1e-6)
            sampled = np.random.normal(fc_w['forecast'].values, std_est)
            if 'max' in mapping:
                sim_feats[mapping['max']] = sampled.max()
            if 'p90' in mapping:
                sim_feats[mapping['p90']] = np.percentile(sampled, 90)
            if 'sum' in mapping:
                sim_feats[mapping['sum']] = sampled.sum()

        feat_vals = [sim_feats.get(f, analysis_df[f].median())
                     for f in significant_features]
        X_sim = np.array(feat_vals).reshape(1, -1)
        mc_results[h_name].append(rf.predict_proba(X_sim)[0][1])

mc_summary = []
for h_name, probs in mc_results.items():
    probs = np.array(probs)
    mc_summary.append({
        'horizon':         h_name,
        'target_date':     horizons[h_name].strftime('%Y-%m'),
        'mean_prob':       round(np.mean(probs), 3),
        'median_prob':     round(np.median(probs), 3),
        'ci_lower_95':     round(np.percentile(probs, 2.5), 3),
        'ci_upper_95':     round(np.percentile(probs, 97.5), 3),
        'prob_above_0.5':  round(np.mean(probs > 0.5), 3),
        'risk_category':   ('HIGH'     if np.mean(probs) > 0.6 else
                            'MODERATE' if np.mean(probs) > 0.4 else 'LOW'),
    })

mc_df = pd.DataFrame(mc_summary)
print("\n=== MONTE CARLO UNCERTAINTY-AWARE PREDICTIONS ===")
print(mc_df.to_string(index=False))
</VSCode.Cell>

### Section 7.6a — Actual Shoreline Retreat Predictions (Physical Units)

Translating erosion **probability** into **expected shoreline retreat in meters**
using the EPR statistics of eroding transects. This bridges the gap between
statistical modeling and actionable coastal management information.

**Method:** Weighted expected value  
$E[\text{retreat}] = P(\text{erosion}) \times \bar{EPR}_{erosion} + (1 - P(\text{erosion})) \times \bar{EPR}_{stable}$

The 95% prediction interval combines SARIMA forecast uncertainty (via MC) with
the natural variability of EPR within each class.

In [ ]:
# =============================================================================
# Section 7.6a — Shoreline retreat prediction in meters
# =============================================================================

# EPR statistics from DSAS data
epr_erosion_mean = dsas_df[dsas_df['erosion_flag'] == 1]['EPR'].mean()
epr_erosion_std  = dsas_df[dsas_df['erosion_flag'] == 1]['EPR'].std()
epr_stable_mean  = dsas_df[dsas_df['erosion_flag'] == 0]['EPR'].mean()
epr_stable_std   = dsas_df[dsas_df['erosion_flag'] == 0]['EPR'].std()
epr_all_mean     = dsas_df['EPR'].mean()

print("=== EPR Distribution Statistics (m/yr) ===")
print(f"  Eroding transects:     mean = {epr_erosion_mean:.3f}, std = {epr_erosion_std:.3f}")
print(f"  Stable/accreting:      mean = {epr_stable_mean:.3f},  std = {epr_stable_std:.3f}")
print(f"  All transects:         mean = {epr_all_mean:.3f}")

# For each horizon, compute expected retreat
retreat_predictions = []

for _, mc_row in mc_df.iterrows():
    h_name = mc_row['horizon']
    h_date = horizons[h_name]
    months_ahead = int(((h_date.year - last_date.year) * 12 +
                         h_date.month - last_date.month))
    years_ahead = months_ahead / 12.0

    p_erosion = mc_row['mean_prob']
    p_lo = mc_row['ci_lower_95']
    p_hi = mc_row['ci_upper_95']

    # Expected EPR (weighted combination)
    expected_epr = p_erosion * epr_erosion_mean + (1 - p_erosion) * epr_stable_mean

    # Expected cumulative retreat over the horizon period
    expected_retreat = expected_epr * years_ahead

    # Uncertainty: propagate both probability CI and EPR variability
    # Best case (low erosion probability, favorable EPR)
    retreat_best = (p_lo * epr_erosion_mean + (1 - p_lo) * epr_stable_mean) * years_ahead
    # Worst case (high erosion probability, adverse EPR)
    retreat_worst = (p_hi * (epr_erosion_mean - epr_erosion_std) +
                     (1 - p_hi) * epr_stable_mean) * years_ahead

    # Monsoon year of forecast
    monsoon_yr = h_date.year if h_date.month >= 4 else h_date.year - 1

    retreat_predictions.append({
        'horizon':              h_name,
        'target_date':          h_date.strftime('%Y-%m'),
        'monsoon_year':         monsoon_yr,
        'months_ahead':         months_ahead,
        'erosion_probability':  round(p_erosion, 3),
        'expected_epr_m_yr':    round(expected_epr, 3),
        'expected_retreat_m':   round(expected_retreat, 3),
        'retreat_best_m':       round(retreat_best, 3),
        'retreat_worst_m':      round(retreat_worst, 3),
        'risk_category':        mc_row['risk_category'],
        'action_level':         ('IMMEDIATE ACTION' if p_erosion > 0.7 else
                                 'ENHANCED MONITORING' if p_erosion > 0.5 else
                                 'ROUTINE MONITORING' if p_erosion > 0.3 else
                                 'STANDARD OPERATIONS'),
    })

retreat_df = pd.DataFrame(retreat_predictions)

print("\n=== ACTUAL SHORELINE RETREAT PREDICTIONS ===")
print(retreat_df[['horizon', 'target_date', 'erosion_probability',
                  'expected_epr_m_yr', 'expected_retreat_m',
                  'retreat_best_m', 'retreat_worst_m',
                  'risk_category', 'action_level']].to_string(index=False))

# Per-horizon driver analysis (which factors are driving the risk)
print("\n=== FORECAST DRIVER ANALYSIS (per horizon) ===")
for _, row in retreat_df.iterrows():
    h_name = row['horizon']
    h_date = horizons[h_name]
    h_feats = horizon_features[horizon_features['horizon'] == row['target_date']]
    if len(h_feats) == 0:
        continue
    h_feats = h_feats.iloc[0]
    print(f"\n  {h_name} ({row['target_date']}):")
    print(f"    Erosion probability: {row['erosion_probability']:.1%}")
    print(f"    Expected retreat:    {row['expected_retreat_m']:.2f} m "
          f"[{row['retreat_worst_m']:.2f} to {row['retreat_best_m']:.2f}]")
    print(f"    Action level:        {row['action_level']}")
    for feat in significant_features[:5]:
        val = h_feats.get(feat, np.nan)
        if feat in individual_thresholds and not np.isnan(val):
            thresh = individual_thresholds[feat]['threshold_all']
            exceeds = val > thresh
            pct = stats.percentileofscore(analysis_df[feat].values, val)
            flag = ' ▲ EXCEEDS' if exceeds else ''
            print(f"    {feat:30s}: {val:>8.3f}  (thresh={thresh:.3f}, "
                  f"P{pct:.0f}){flag}")

retreat_df.to_csv('./outputs/retreat_predictions.csv', index=False)

In [ ]:
# =============================================================================
# Section 7.6b — Per-transect vulnerability scoring
# =============================================================================

# Combine historical erosion severity with forecast probability
# to produce a transect-level risk score

print("=== PER-TRANSECT VULNERABILITY SCORING ===\n")

# Historical vulnerability (from DSAS)
dsas_scored = dsas_df[['id', 'EPR', 'NSM', 'SCE', 'epr_class', 'erosion_flag']].copy()

# Map EPR class to historical severity score (0-4)
severity_map = {
    'eroded_high': 4, 'eroded_low': 3,
    'stable': 2, 'accreted_low': 1, 'accreted_high': 0
}
dsas_scored['hist_severity'] = dsas_scored['epr_class'].map(severity_map)

# Normalize EPR to 0-1 vulnerability index (more negative = more vulnerable)
epr_min, epr_max = dsas_scored['EPR'].min(), dsas_scored['EPR'].max()
dsas_scored['epr_vulnerability'] = (epr_max - dsas_scored['EPR']) / (epr_max - epr_min)

# For each forecast horizon, compute transect-level risk
for _, row in retreat_df.iterrows():
    p_ero = row['erosion_probability']
    # Combined risk = historical vulnerability × forecast probability
    col_name = f"risk_{row['horizon']}"
    dsas_scored[col_name] = dsas_scored['epr_vulnerability'] * p_ero
    # Risk category
    cat_name = f"cat_{row['horizon']}"
    dsas_scored[cat_name] = pd.cut(
        dsas_scored[col_name],
        bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
        labels=['Very Low', 'Low', 'Moderate', 'High', 'Very High'],
        include_lowest=True
    )

# Summary: most vulnerable transects
print("Top 10 most vulnerable transects (H12 forecast):")
h12_col = 'risk_H12' if 'risk_H12' in dsas_scored.columns else dsas_scored.columns[-2]
top10 = dsas_scored.nlargest(10, h12_col)[
    ['id', 'EPR', 'epr_class', 'epr_vulnerability', h12_col]
]
print(top10.to_string(index=False))

# Risk category distribution per horizon
print("\n--- Transect Risk Distribution per Horizon ---")
for _, row in retreat_df.iterrows():
    cat_col = f"cat_{row['horizon']}"
    if cat_col in dsas_scored.columns:
        dist = dsas_scored[cat_col].value_counts().sort_index()
        print(f"\n  {row['horizon']} ({row['target_date']}):")
        for cat, cnt in dist.items():
            pct = cnt / len(dsas_scored) * 100
            print(f"    {cat:12s}: {cnt:3d} transects ({pct:.1f}%)")

# Vulnerability map plot
fig, axes = plt.subplots(1, min(4, len(retreat_df)), figsize=(16, 5))
if len(retreat_df) <= 1:
    axes = [axes]

vuln_cmap = plt.cm.RdYlGn_r
for ax, (_, row) in zip(axes, retreat_df.iterrows()):
    risk_col = f"risk_{row['horizon']}"
    if risk_col not in dsas_scored.columns:
        continue
    sc = ax.scatter(dsas_scored['id'], dsas_scored[risk_col],
                    c=dsas_scored[risk_col], cmap=vuln_cmap,
                    vmin=0, vmax=1, s=20, edgecolors='none')
    ax.axhline(0.5, color='red', ls='--', lw=1, alpha=0.5)
    ax.set_xlabel('Transect ID')
    ax.set_ylabel('Combined Risk Score')
    ax.set_title(f"{row['horizon']} ({row['target_date']})\n"
                 f"P(erosion) = {row['erosion_probability']:.0%}")
    ax.grid(True, alpha=0.3)

plt.suptitle('Per-Transect Vulnerability Score (historical × forecast)', fontsize=13)
plt.tight_layout()
plt.savefig('./figures/transect_vulnerability.png', dpi=150, bbox_inches='tight')
plt.show()

dsas_scored.to_csv('./outputs/transect_vulnerability_scores.csv', index=False)

In [ ]:
# =============================================================================
# Section 7.6c — Forecast skill vs climatological baseline
# =============================================================================

# Climatological baseline: predict erosion probability = historical base rate
base_rate = analysis_df['erosion_label'].mean()
print(f"=== FORECAST SKILL vs CLIMATOLOGY BASELINE ===")
print(f"Historical base rate (P_erosion): {base_rate:.3f}")

# Brier Skill Score: BSS = 1 - BS_forecast / BS_climatology
# BS = mean((forecast_prob - actual)^2)

if len(hc_df) > 0:
    bs_forecast = np.mean((hc_df['probability'].values - hc_df['actual'].values)**2)
    bs_clim     = np.mean((base_rate - hc_df['actual'].values)**2)
    bss = 1 - bs_forecast / bs_clim if bs_clim > 0 else 0

    # ROC AUC for hindcast
    if len(np.unique(hc_df['actual'])) >= 2:
        hc_fpr, hc_tpr, _ = roc_curve(hc_df['actual'], hc_df['probability'])
        hc_auc = auc(hc_fpr, hc_tpr)
    else:
        hc_auc = np.nan

    print(f"\nBrier Score (forecast):   {bs_forecast:.4f}")
    print(f"Brier Score (climatology): {bs_clim:.4f}")
    print(f"Brier Skill Score (BSS):   {bss:.4f}")
    print(f"  BSS > 0 means forecast is better than always predicting base rate")
    print(f"  BSS interpretation: {'SKILLFUL' if bss > 0 else 'NO SKILL vs climatology'}")
    print(f"\nHindcast AUC: {hc_auc:.3f}" if not np.isnan(hc_auc) else "Hindcast AUC: N/A")

    # Reliability table: bin predictions, compare with observed frequency
    n_bins = 5
    bins = np.linspace(0, 1, n_bins + 1)
    hc_df_copy = hc_df.copy()
    hc_df_copy['prob_bin'] = pd.cut(hc_df_copy['probability'], bins=bins,
                                     include_lowest=True)
    rel_table = hc_df_copy.groupby('prob_bin', observed=False).agg(
        n_forecasts=('actual', 'count'),
        mean_predicted=('probability', 'mean'),
        observed_freq=('actual', 'mean'),
    ).dropna()

    print("\n--- Reliability Table ---")
    print(rel_table.to_string())

    # Plot skill comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Left: Brier Score decomposition
    labels = ['Forecast', 'Climatology']
    values = [bs_forecast, bs_clim]
    colors_bs = ['#185FA5' if bss > 0 else '#E24B4A', '#888780']
    ax1.bar(labels, values, color=colors_bs, edgecolor='white')
    ax1.set_ylabel('Brier Score (lower = better)')
    ax1.set_title(f'Brier Skill Score = {bss:.3f}\n'
                  f'({"Forecast beats climatology" if bss > 0 else "Climatology is better"})')
    ax1.grid(True, alpha=0.3, axis='y')
    for i, (lbl, val) in enumerate(zip(labels, values)):
        ax1.text(i, val + 0.01, f'{val:.3f}', ha='center', fontsize=11)

    # Right: reliability diagram
    if len(rel_table) > 0 and not rel_table['mean_predicted'].isna().all():
        ax2.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect reliability')
        ax2.scatter(rel_table['mean_predicted'], rel_table['observed_freq'],
                    s=rel_table['n_forecasts'] * 30, c='#185FA5', zorder=5,
                    edgecolors='white')
        ax2.set_xlabel('Mean predicted probability')
        ax2.set_ylabel('Observed frequency')
        ax2.set_title('Reliability Diagram')
        ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
        ax2.legend(fontsize=9)
        ax2.grid(True, alpha=0.3)
    else:
        ax2.text(0.5, 0.5, 'Insufficient data\nfor reliability diagram',
                 ha='center', va='center', fontsize=12)
        ax2.set_title('Reliability Diagram')

    plt.tight_layout()
    plt.savefig('./figures/forecast_skill.png', dpi=150)
    plt.show()
else:
    print("\nNo hindcast results available for skill assessment.")

In [ ]:
# =============================================================================
# Section 7.7 — Month-by-month risk timeline (24 months)
# =============================================================================

monthly_risk = []
ref_fc = sarima_forecasts['Hm0_max']

for _, fc_row in ref_fc.iterrows():
    month_date = fc_row['date']
    month_str  = month_date.strftime('%Y-%m')

    month_vals = {}
    for var in ['WindMax', 'Hm0_max', 'UcurrMax', 'StormDays_wave']:
        fc_var = sarima_forecasts[var]
        match  = fc_var[fc_var['date'] == month_date]
        month_vals[var] = match['forecast'].values[0] if len(match) else np.nan

    exceedances = []
    for feat, val in [('wind_max_annual', month_vals.get('WindMax', np.nan)),
                       ('Hm0_max_annual',  month_vals.get('Hm0_max',  np.nan)),
                       ('ucurr_max_annual', month_vals.get('UcurrMax', np.nan))]:
        if feat in individual_thresholds and not np.isnan(val):
            if val > individual_thresholds[feat]['threshold_low']:
                exceedances.append(feat)

    n_exc       = len(exceedances)
    risk_score  = min(n_exc, 3)
    risk_labels = ['Stable', 'Watch', 'Warning', 'Alert']

    monthly_risk.append({
        'month':           month_str,
        'horizon_group':   ('6m'  if month_date <= horizons['H6']  else
                            '12m' if month_date <= horizons['H12'] else
                            '18m' if month_date <= horizons['H18'] else '24m'),
        'wind_forecast':   round(month_vals.get('WindMax', np.nan), 3),
        'wave_forecast':   round(month_vals.get('Hm0_max',  np.nan), 3),
        'current_forecast':round(month_vals.get('UcurrMax', np.nan), 3),
        'factors_exceeding': exceedances,
        'n_exceedances':   n_exc,
        'risk_score':      risk_score,
        'risk_label':      risk_labels[risk_score],
    })

monthly_risk_df = pd.DataFrame(monthly_risk)
print("\n=== MONTH-BY-MONTH EROSION RISK FORECAST ===")
print(monthly_risk_df[['month', 'horizon_group', 'wind_forecast',
                        'wave_forecast', 'current_forecast',
                        'n_exceedances', 'risk_label']].to_string(index=False))
</VSCode.Cell>

In [ ]:
# =============================================================================
# Section 7.8 — Advanced Forecast Dashboard (6-panel publication figure)
# =============================================================================

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.30)

# ── Panel 1: Erosion probability + actual retreat at each horizon ──
ax1 = fig.add_subplot(gs[0, 0])
bar_colors = ['#1D9E75' if r == 'LOW' else
              '#EF9F27' if r == 'MODERATE' else
              '#E24B4A' for r in mc_df['risk_category']]
bars = ax1.bar(mc_df['target_date'], mc_df['mean_prob'],
               color=bar_colors, alpha=0.85, edgecolor='white')
ax1.errorbar(mc_df['target_date'], mc_df['mean_prob'],
             yerr=[mc_df['mean_prob'] - mc_df['ci_lower_95'],
                   mc_df['ci_upper_95'] - mc_df['mean_prob']],
             fmt='none', color='black', capsize=6, linewidth=1.5)
ax1.axhline(0.5, color='red', linestyle='--', linewidth=1.2,
            label='Erosion onset (p=0.5)')
ax1.set_ylim(0, 1)
ax1.set_ylabel('Predicted erosion probability')
ax1.set_title('Erosion risk at 4 forecast horizons\n(error bars = 95% MC CI)')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3, axis='y')
for bar, row in zip(bars, mc_df.itertuples()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.04,
             row.risk_category, ha='center', fontsize=10, fontweight='500')

# ── Panel 2: Actual retreat predictions (meters) ──
ax2 = fig.add_subplot(gs[0, 1])
x_pos = range(len(retreat_df))
retreat_vals = retreat_df['expected_retreat_m'].values
retreat_worst = retreat_df['retreat_worst_m'].values
retreat_best = retreat_df['retreat_best_m'].values

bar_cols_r = ['#E24B4A' if v < -0.5 else '#EF9F27' if v < 0 else '#1D9E75'
              for v in retreat_vals]
ax2.bar(x_pos, retreat_vals, color=bar_cols_r, alpha=0.85, edgecolor='white')
ax2.errorbar(x_pos, retreat_vals,
             yerr=[retreat_vals - retreat_worst,
                   retreat_best - retreat_vals],
             fmt='none', color='black', capsize=6, linewidth=1.5)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(retreat_df['target_date'], rotation=30)
ax2.axhline(0, color='gray', ls='-', lw=0.8)
ax2.set_ylabel('Expected shoreline change (m)')
ax2.set_title('Predicted shoreline retreat by horizon\n(negative = erosion)')
ax2.grid(True, alpha=0.3, axis='y')
for i, (val, act) in enumerate(zip(retreat_vals, retreat_df['action_level'])):
    ax2.text(i, val - 0.05, f'{val:.2f}m', ha='center', fontsize=9, fontweight='500')

# ── Panel 3: Monthly risk score heatmap ──
ax3 = fig.add_subplot(gs[1, 0])
risk_matrix = monthly_risk_df[['month', 'risk_score']].copy()
risk_matrix['month_idx'] = range(len(risk_matrix))
risk_array = risk_matrix['risk_score'].values.reshape(1, -1)
im3 = ax3.imshow(risk_array, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=3)
ax3.set_yticks([])
ax3.set_xticks(range(0, len(risk_matrix), 3))
ax3.set_xticklabels(risk_matrix['month'].iloc[::3], rotation=60, fontsize=7)
ax3.set_title('Monthly erosion risk score — 24 month timeline')
ax3.set_xlabel('Month')
plt.colorbar(im3, ax=ax3, label='Risk score (0=stable, 3=alert)', shrink=0.8)

# ── Panel 4: Wave + Wind forecasts with threshold lines ──
ax4 = fig.add_subplot(gs[1, 1])
ax4_r = ax4.twinx()
fc_wave = sarima_forecasts['Hm0_max']
fc_wind = sarima_forecasts['WindMax']

ax4.fill_between(fc_wave['date'], fc_wave['lower_95'], fc_wave['upper_95'],
                 alpha=0.15, color='#185FA5')
ax4.plot(fc_wave['date'], fc_wave['forecast'],
         color='#185FA5', linewidth=2, label='Hm0 forecast')
ax4_r.fill_between(fc_wind['date'], fc_wind['lower_95'], fc_wind['upper_95'],
                   alpha=0.12, color='#E24B4A')
ax4_r.plot(fc_wind['date'], fc_wind['forecast'],
           color='#E24B4A', linewidth=2, linestyle='--', label='Wind forecast')

if 'Hm0_max_annual' in individual_thresholds:
    ax4.axhline(individual_thresholds['Hm0_max_annual']['threshold_low'],
                color='#185FA5', linestyle=':', linewidth=1.2,
                label='Wave erosion threshold')
    if not np.isnan(individual_thresholds['Hm0_max_annual']['threshold_high']):
        ax4.axhline(individual_thresholds['Hm0_max_annual']['threshold_high'],
                    color='navy', linestyle='-.', linewidth=1.2)

for h_name, h_date in horizons.items():
    ax4.axvline(h_date, color=horizon_colors[h_name], linestyle=':', linewidth=1)

ax4.set_ylabel('Hm0 max (m)', color='#185FA5')
ax4_r.set_ylabel('Wind max (m/s)', color='#E24B4A')
ax4.set_title('Forcing forecasts with erosion thresholds')
lines1, lbl1 = ax4.get_legend_handles_labels()
lines2, lbl2 = ax4_r.get_legend_handles_labels()
ax4.legend(lines1 + lines2, lbl1 + lbl2, fontsize=7, loc='upper left')
ax4.grid(True, alpha=0.3)

# ── Panel 5: Per-transect vulnerability (H12) ──
ax5 = fig.add_subplot(gs[2, 0])
h12_risk = 'risk_H12' if 'risk_H12' in dsas_scored.columns else None
if h12_risk:
    sc5 = ax5.scatter(dsas_scored['id'], dsas_scored[h12_risk],
                      c=dsas_scored[h12_risk], cmap='RdYlGn_r',
                      vmin=0, vmax=1, s=25, edgecolors='none')
    ax5.axhline(0.5, color='red', ls='--', lw=1, alpha=0.5)
    ax5.set_xlabel('Transect ID')
    ax5.set_ylabel('Combined Risk Score')
    ax5.set_title('Per-transect vulnerability — 12-month horizon')
    plt.colorbar(sc5, ax=ax5, label='Risk (0=safe, 1=critical)', shrink=0.8)
    ax5.grid(True, alpha=0.3)
else:
    ax5.text(0.5, 0.5, 'Transect vulnerability\nnot available', ha='center', va='center')

# ── Panel 6: Actionable summary table ──
ax6 = fig.add_subplot(gs[2, 1])
ax6.axis('off')
table_data = []
for _, row in retreat_df.iterrows():
    table_data.append([
        row['horizon'],
        row['target_date'],
        f"{row['erosion_probability']:.0%}",
        f"{row['expected_retreat_m']:.2f}",
        f"[{row['retreat_worst_m']:.2f}, {row['retreat_best_m']:.2f}]",
        row['risk_category'],
        row['action_level'],
    ])
col_labels = ['Horizon', 'Date', 'P(erosion)', 'Retreat\n(m)',
              '95% Range\n(m)', 'Risk', 'Action']
table = ax6.table(cellText=table_data, colLabels=col_labels,
                   cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.6)
# Color code risk cells
for i, row in enumerate(table_data):
    risk = row[5]
    color = ('#FFE0E0' if risk == 'HIGH' else
             '#FFF3D0' if risk == 'MODERATE' else '#E0F5E0')
    for j in range(len(col_labels)):
        table[i + 1, j].set_facecolor(color)
ax6.set_title('ACTIONABLE PREDICTION SUMMARY', fontsize=12, fontweight='bold', pad=20)

plt.suptitle('24-Month Coastal Erosion Risk Forecast — Advanced Research Dashboard',
             fontsize=16, fontweight='bold', y=0.98)
plt.savefig('./figures/forecast_dashboard_advanced.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# Section 7.9 — Save all outputs (comprehensive)
# =============================================================================

mc_df.to_csv('./outputs/erosion_forecast_horizons.csv',   index=False)
monthly_risk_df.to_csv('./outputs/erosion_forecast_monthly.csv', index=False)

horizon_export = horizon_features.merge(
    mc_df[['target_date', 'mean_prob', 'ci_lower_95', 'ci_upper_95', 'risk_category']],
    left_on='horizon', right_on='target_date', how='left'
)
horizon_export.to_csv('./outputs/erosion_forecast_features.csv', index=False)

# Additional outputs from enhanced pipeline
retreat_df.to_csv('./outputs/retreat_predictions.csv', index=False)
dsas_scored.to_csv('./outputs/transect_vulnerability_scores.csv', index=False)
sarima_diag_df.to_csv('./outputs/sarima_model_diagnostics.csv', index=False)

print("=" * 60)
print("  ALL OUTPUTS SAVED SUCCESSFULLY")
print("=" * 60)
print("\n--- Threshold Outputs ---")
print("  ./outputs/multi_method_thresholds.csv")
print("  ./outputs/erosion_thresholds_master.csv")
print("  ./outputs/erosion_event_diagnosis.csv")
print("\n--- Forecast Outputs ---")
print("  ./outputs/sarima_model_diagnostics.csv    (AIC grid search, residual tests)")
print("  ./outputs/hindcast_validation.csv          (temporal cross-validation)")
print("  ./outputs/erosion_forecast_horizons.csv    (MC probability at 4 horizons)")
print("  ./outputs/erosion_forecast_monthly.csv     (month-by-month risk timeline)")
print("  ./outputs/erosion_forecast_features.csv    (aggregated features + risk)")
print("  ./outputs/retreat_predictions.csv          (shoreline retreat in METERS)")
print("  ./outputs/transect_vulnerability_scores.csv(per-transect risk scores)")
print("\n--- Figures ---")
print("  ./figures/  (all publication-quality plots)")

print("\n" + "=" * 60)
print("  PREDICTION SUMMARY")
print("=" * 60)
for _, row in retreat_df.iterrows():
    print(f"\n  {row['horizon']} → {row['target_date']}:")
    print(f"    Erosion probability:  {row['erosion_probability']:.0%}")
    print(f"    Expected retreat:     {row['expected_retreat_m']:.2f} m "
          f"[{row['retreat_worst_m']:.2f} to {row['retreat_best_m']:.2f}]")
    print(f"    Risk:                 {row['risk_category']}")
    print(f"    Recommended action:   {row['action_level']}")

In [ ]:

# =============================================================================
# AUTO-GENERATED: Comprehensive JSON export for frontend
# =============================================================================
import json, os, numpy as np, pandas as pd

_RESULTS_PATH = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\results\results_51648a295218.json"
_FRONTEND_PATH = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\frontend\public\data"

def _safe(v):
    """Make a value JSON-serialisable."""
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, np.ndarray):
        return v.tolist()
    if isinstance(v, pd.Timestamp):
        return v.isoformat()
    if isinstance(v, (pd.Series, pd.Index)):
        return v.tolist()
    if hasattr(v, 'item'):
        return v.item()
    return v

# ---- Build the results dict ----
results = {}

# =====================================================================
# 1. Summary statistics
# =====================================================================
try:
    results["summary"] = {
        "totalTransects": _safe(transect_stats.get("Total_Transects", 0)),
        "erodingTransects": _safe(int(transect_stats.get("Pct_Eroding", 0) / 100 * transect_stats.get("Total_Transects", 0))),
        "erosionRate": _safe(round(transect_stats.get("Pct_Eroding", 0), 1)),
        "meanNSM": _safe(round(transect_stats.get("Mean_NSM", 0), 2)),
        "medianNSM": _safe(round(transect_stats.get("Median_NSM", 0), 2)),
        "totalYears": _safe(len(analysis_df)),
        "erosionYears": _safe(int(analysis_df["erosion_label"].sum())),
        "erosionYearsList": [int(y) for y in sorted(all_erosion_years)],
        "analysisYearRange": f"{int(analysis_df['monsoon_year'].min())}-{int(analysis_df['monsoon_year'].max())}",
        "meanEPR": _safe(round(transect_stats.get("Mean_EPR", 0), 2)),
        "significantFeatures": list(significant_features),
    }
except Exception as e:
    print(f"Summary export error: {e}")
    results["summary"] = {}

# =====================================================================
# 2. Shoreline transect data
# =====================================================================
try:
    results["shoreline"] = dsas_df[['id', 'EPR', 'NSM', 'SCE', 'epr_class', 'erosion_flag']].to_dict(orient='records')
except Exception as e:
    print(f"Shoreline export error: {e}")
    results["shoreline"] = []

# =====================================================================
# 3. Annual analysis data (time series)
# =====================================================================
try:
    ts_cols = [c for c in analysis_df.columns if c not in ['geometry']]
    results["timeSeries"] = analysis_df[ts_cols].to_dict(orient='records')
except Exception as e:
    print(f"TimeSeries export error: {e}")
    results["timeSeries"] = []

# =====================================================================
# 4. Mann-Whitney statistical tests
# =====================================================================
try:
    results["statisticalTests"] = mw_df.to_dict(orient='records')
except Exception as e:
    print(f"Statistical tests export error: {e}")
    results["statisticalTests"] = []

# =====================================================================
# 5. Four-method ensemble thresholds
# =====================================================================
try:
    threshold_export = []
    for feat in significant_features:
        t = individual_thresholds.get(feat, {})
        e = ensemble_results.get(feat, {})
        row = {
            "feature": feat,
            "thresholdAll": _safe(t.get('threshold_all')),
            "thresholdLow": _safe(t.get('threshold_low')),
            "thresholdHigh": _safe(t.get('threshold_high')),
            "pValue": _safe(t.get('p_value')),
            "methods": {},
        }
        for method_name in ['roc_youden', 'bayesian_logistic', 'change_point', 'mutual_info']:
            if method_name in e:
                m = e[method_name]
                row["methods"][method_name] = {
                    "threshold": _safe(m.get('threshold')),
                    "ci_lower": _safe(m.get('ci_lower')),
                    "ci_upper": _safe(m.get('ci_upper')),
                    "statistic": _safe(m.get('statistic', m.get('auc', m.get('mi')))),
                    "pValue": _safe(m.get('p_value')),
                    "significant": bool(m.get('significant', True)),
                }
        threshold_export.append(row)
    results["thresholds"] = threshold_export
except Exception as e:
    print(f"Threshold export error: {e}")
    results["thresholds"] = []

# =====================================================================
# 6. Random Forest model
# =====================================================================
try:
    importances = pd.Series(rf.feature_importances_, index=significant_features).sort_values(ascending=False)
    results["rfModel"] = {
        "featureImportance": [
            {"feature": feat, "importance": _safe(round(imp, 4))}
            for feat, imp in importances.items()
        ],
        "oobScore": _safe(round(rf.oob_score_, 4)) if hasattr(rf, 'oob_score_') else None,
        "nEstimators": _safe(rf.n_estimators),
    }
except Exception as e:
    print(f"RF model export error: {e}")
    results["rfModel"] = None

# =====================================================================
# 7. SARIMA diagnostics (AIC grid search)
# =====================================================================
try:
    results["sarimaDiagnostics"] = sarima_diag_df.to_dict(orient='records')
except Exception as e:
    print(f"SARIMA diagnostics export error: {e}")
    results["sarimaDiagnostics"] = []

# =====================================================================
# 8. SARIMA forecasts (monthly per variable)
# =====================================================================
try:
    fc_export = {}
    for var_name, fc_df_var in sarima_forecasts.items():
        fc_export[var_name] = {
            "monthly": fc_df_var.to_dict(orient='records'),
        }
    results["sarimaForecasts"] = fc_export
except Exception as e:
    print(f"SARIMA forecasts export error: {e}")
    results["sarimaForecasts"] = {}

# =====================================================================
# 9. Hindcast validation
# =====================================================================
try:
    if len(hc_df) > 0:
        hits   = int(((hc_df['actual'] == 1) & (hc_df['predicted'] == 1)).sum())
        misses = int(((hc_df['actual'] == 1) & (hc_df['predicted'] == 0)).sum())
        false_alarms = int(((hc_df['actual'] == 0) & (hc_df['predicted'] == 1)).sum())
        correct_rej  = int(((hc_df['actual'] == 0) & (hc_df['predicted'] == 0)).sum())
        pod = hits / max(hits + misses, 1)
        far = false_alarms / max(hits + false_alarms, 1)
        csi = hits / max(hits + misses + false_alarms, 1)
        accuracy = (hits + correct_rej) / len(hc_df)
        results["hindcast"] = {
            "results": hc_df.to_dict(orient='records'),
            "metrics": {
                "accuracy": _safe(round(accuracy, 4)),
                "pod": _safe(round(pod, 4)),
                "far": _safe(round(far, 4)),
                "csi": _safe(round(csi, 4)),
                "hits": hits,
                "misses": misses,
                "falseAlarms": false_alarms,
                "correctRejections": correct_rej,
                "totalYears": len(hc_df),
            },
        }
    else:
        results["hindcast"] = None
except Exception as e:
    print(f"Hindcast export error: {e}")
    results["hindcast"] = None

# =====================================================================
# 10. Monte Carlo results
# =====================================================================
try:
    results["monteCarlo"] = {
        "nSimulations": 2000,
        "horizons": mc_df.to_dict(orient='records'),
    }
except Exception as e:
    print(f"Monte Carlo export error: {e}")
    results["monteCarlo"] = None

# =====================================================================
# 11. Retreat predictions (actual meters)
# =====================================================================
try:
    results["retreatPredictions"] = retreat_df.to_dict(orient='records')
except Exception as e:
    print(f"Retreat predictions export error: {e}")
    results["retreatPredictions"] = []

# =====================================================================
# 12. Per-transect vulnerability scores
# =====================================================================
try:
    vuln_cols = [c for c in dsas_scored.columns if c.startswith('risk_') or c.startswith('cat_')]
    base_cols = ['id', 'EPR', 'NSM', 'epr_class', 'epr_vulnerability']
    export_cols = [c for c in base_cols + vuln_cols if c in dsas_scored.columns]
    results["transectVulnerability"] = {
        "data": dsas_scored[export_cols].to_dict(orient='records'),
        "summary": {},
    }
    for col in vuln_cols:
        if col.startswith('cat_') and col in dsas_scored.columns:
            h_name = col.replace('cat_', '')
            dist = dsas_scored[col].value_counts().to_dict()
            results["transectVulnerability"]["summary"][h_name] = {
                str(k): int(v) for k, v in dist.items()
            }
except Exception as e:
    print(f"Transect vulnerability export error: {e}")
    results["transectVulnerability"] = None

# =====================================================================
# 13. Forecast skill scores
# =====================================================================
try:
    base_rate_val = float(analysis_df['erosion_label'].mean())
    if len(hc_df) > 0:
        bs_fc = float(np.mean((hc_df['probability'].values - hc_df['actual'].values)**2))
        bs_clim = float(np.mean((base_rate_val - hc_df['actual'].values)**2))
        bss_val = 1 - bs_fc / bs_clim if bs_clim > 0 else 0
        results["forecastSkill"] = {
            "baseRate": _safe(round(base_rate_val, 4)),
            "brierScoreForecast": _safe(round(bs_fc, 4)),
            "brierScoreClimatology": _safe(round(bs_clim, 4)),
            "brierSkillScore": _safe(round(bss_val, 4)),
            "skillful": bss_val > 0,
        }
    else:
        results["forecastSkill"] = None
except Exception as e:
    print(f"Forecast skill export error: {e}")
    results["forecastSkill"] = None

# =====================================================================
# 14. Monthly risk timeline
# =====================================================================
try:
    results["monthlyRisk"] = monthly_risk_df.to_dict(orient='records')
except Exception as e:
    print(f"Monthly risk export error: {e}")
    results["monthlyRisk"] = []

# =====================================================================
# 15. Horizon features (aggregated annual per horizon)
# =====================================================================
try:
    results["horizonFeatures"] = horizon_features.to_dict(orient='records')
except Exception as e:
    print(f"Horizon features export error: {e}")
    results["horizonFeatures"] = []

# =====================================================================
# 16. Erosion predictions per horizon (RF probabilities)
# =====================================================================
try:
    results["erosionPredictions"] = erosion_predictions
except Exception as e:
    print(f"Erosion predictions export error: {e}")
    results["erosionPredictions"] = []

# ---- Write JSON ----
os.makedirs(os.path.dirname(_RESULTS_PATH), exist_ok=True)
with open(_RESULTS_PATH, "w") as _f:
    json.dump(results, _f, indent=2, default=str)

os.makedirs(_FRONTEND_PATH, exist_ok=True)
with open(os.path.join(_FRONTEND_PATH, "analysis_results.json"), "w") as _f:
    json.dump(results, _f, indent=2, default=str)

print(f"✓ Results exported to {_RESULTS_PATH}")
print(f"✓ Results copied to {_FRONTEND_PATH}/analysis_results.json")

